# Utility-Privacy Audit of Diffusion-Based Minority-Class Augmentation for Fraud Detection (Revision 1)

Companion notebook for the revised manuscript submitted to *Emerging Science Journal*. It keeps the original
pipeline (Gaussian DDPM on the fraud class; ROS, SMOTE, ADASYN and CTGAN baselines; four classifiers;
loss-based and nearest-neighbour membership-inference attacks; DP-SGD) and adds every analysis requested in
the first round of review.

| Reviewer request | Where it is handled |
|---|---|
| R2.1 Run the nearest-neighbour attack on ROS, SMOTE and ADASYN too; state what is released | Part A (privacy block), Part G, Table 5 |
| R2.2 Confidence intervals for the attack AUCs over several seeds | Generators are re-trained for every seed; bootstrap and DeLong CIs; permutation chance band |
| R2.3 Separate data scale from training length; held-out-loss early stopping | Part D (IEEE-CIS fraud subsampling at n = 344, 1000, 2000, 4000 with fixed architecture, AUC against epochs, three stopping rules) |
| R2.3 Non-DP lightweight reference; matched-architecture DP sweep | Part C |
| R2.4 Select r* on a validation partition, not on the test set | Parts A, E, F: stratified fit/validation split inside the training partition |
| R2.4 Paired tests against every method with multiplicity correction | Part G (Wilcoxon signed-rank with Holm correction and paired bootstrap CIs) |
| R2.4 Threshold effects | F1 and MCC at the stated 0.5 threshold and at a validation-tuned threshold; AUPRC and ROC-AUC with SDs |
| R2.6 Missing hyperparameters, reproducibility | Everything is in `CONFIG`; Part G exports the hyperparameter table |
| R2.6 IEEE-CIS: more seeds, SDs | Part E (5 confirmation seeds, mean and SD for every metric) |
| R2.6 Time-ordered split for ULB | Part F |
| R1 Figures readable (2x2 grids), editable chart files | Part G regenerates every chart from CSV and writes `figure_source_data.xlsx` with native Excel charts |

**How to run on Kaggle.** Attach `mlg-ulb/creditcardfraud` and the `ieee-fraud-detection` competition data, turn the
GPU on, and use *Save Version -> Run All*. Estimated GPU time (from the per-fit timings of the submitted run):
Part A about 5 h, Part C about 1 h, Part D about 1.7 h, Part E about 1.5 h, Part F about 0.5 h. A single session may
hit Kaggle's 12-hour limit, so two sessions are recommended:

* **Session 1:** `A_ulb_main`, `F_temporal`, `G_assemble` switched on (about 5.5 h).
* **Session 2:** download `revision_results/` from session 1, upload it as a private Kaggle dataset, attach it, then
  switch on `C_dp`, `D_scale`, `E_ieee`, `G_assemble` (about 4.5 h). Part G reads results from both sessions.

A hard budget guard (`max_total_runtime_hours`) stops any loop cleanly at a seed boundary and every finished seed
is saved immediately, so a rerun resumes where it stopped. Run once with `RUN_MODE = "quick_test"` first
(about 15-25 minutes) to check the environment.

**Outputs used by the manuscript:** `revision_results/manuscript_values.json` (every number quoted in the revised
manuscript, keyed by placeholder name), `revision_results/tables/*.csv`, `revision_figures/*.png|pdf` and
`revision_results/figure_source_data.xlsx`.

## 0. Setup

In [1]:
# ============================================================
# 0. Install dependencies (Kaggle images already ship torch, sklearn, xgboost, pandas)
# ============================================================
import sys, subprocess, importlib

def pip_install(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
        return True
    except Exception:
        pass
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        return True
    except Exception as e:
        print(f"[warn] could not install {pkg}: {e}")
        return False

for pkg, mod in [("imbalanced-learn", "imblearn"), ("xgboost", "xgboost"), ("shap", "shap"),
                 ("xlsxwriter", "xlsxwriter")]:
    pip_install(pkg, mod)
HAS_CTGAN = pip_install("ctgan", "ctgan")
HAS_OPACUS = pip_install("opacus", "opacus")
print("ctgan:", HAS_CTGAN, "| opacus:", HAS_OPACUS)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.9/308.9 kB 6.2 MB/s eta 0:00:00
ctgan: True | opacus: True


In [2]:
# ============================================================
# 1. Imports, run configuration, output folders
# ============================================================
import os, glob, math, json, warnings, random, time, copy, itertools
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (precision_score, recall_score, f1_score, average_precision_score,
                             matthews_corrcoef, roc_auc_score, precision_recall_curve)
from scipy import stats
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN

try:
    import xgboost
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ------------------------------------------------------------------
# Which parts to run. Parts are independent; Part G assembles whatever results exist.
# ------------------------------------------------------------------
PARTS = {
    "A_ulb_main": True,        # ULB: validation-based selection, 10-seed confirmation, all privacy attacks, SHAP, PCA
    "C_dp": True,              # ULB: DP-SGD sweep, both architectures, non-DP references, 5 seeds
    "D_scale": True,           # IEEE-CIS fraud-size subsampling + ULB epoch sweep, three stopping rules
    "E_ieee": True,            # IEEE-CIS: validation-based selection, 5 seeds, all privacy attacks
    "F_temporal": True,        # ULB: time-ordered split robustness check
    "G_assemble": True,        # tables, figures, Excel chart workbook, manuscript_values.json
}

RUN_MODE = "full_paper"   # "quick_test" (smoke test, ~20 min) or "full_paper"

BASE_CONFIG = {
    "seed": 42,
    "test_size": 0.30,                 # stratified train/test split (same as the original submission)
    "val_size": 0.20,                  # share of the training partition held out for selecting r* and the threshold
    "augmentation_ratios": [0.5, 1.0, 2.0, 5.0, 10.0],
    "screen_seeds": [0, 1, 2],         # selection stage (fit -> validation)
    "confirm_seeds": list(range(100, 110)),   # confirmation stage (train -> test), fresh seeds
    "diffusion": {
        "timesteps": 200, "beta_start": 1e-4, "beta_end": 0.02,
        "hidden_dim": 256, "t_dim": 128, "n_blocks": 4,
        "lr": 1e-3, "batch_size": 64, "ema_decay": 0.995,
        "max_epochs": 3000, "min_epochs": 800, "rel_tol": 0.02,   # window w = max(20, max_epochs // 20)
        "loss_repeats": 20,                                       # (t, eps) draws per record for the loss attack
    },
    "ctgan": {"epochs": 100, "batch_size": 500, "embedding_dim": 128, "generator_dim": (256, 256),
              "discriminator_dim": (256, 256), "generator_lr": 2e-4, "discriminator_lr": 2e-4,
              "generator_decay": 1e-6, "discriminator_decay": 1e-6, "discriminator_steps": 1, "pac": 10},
    "resampling": {"k_neighbors": 5, "maj_ref_size": 20000},
    "classifiers": {
        "lr_max_iter": 2000, "lr_C": 1.0,
        "rf_n_estimators": 100, "rf_max_depth": 8, "rf_min_samples_leaf": 5,
        "xgb_n_estimators": 100, "xgb_max_depth": 6, "xgb_learning_rate": 0.1,
        "xgb_subsample": 0.8, "xgb_colsample_bytree": 0.8,
        "mlp_hidden": (64, 32), "mlp_epochs": 30, "mlp_batch_size": 2048, "mlp_lr": 1e-3,
        "decision_threshold": 0.5,
    },
    "mia": {"nn_k": [1, 5], "fixed_ref_size": 5000, "n_boot": 1000, "n_perm": 2000},
    "dp": {"noise_multipliers": [0.5, 1.0, 2.0, 4.0], "max_grad_norm": 1.0, "delta": 1e-5, "epochs": 250,
           "seeds": [200, 201, 202, 203, 204],
           "light_hidden_dim": 64, "light_t_dim": 32, "light_n_blocks": 2},
    "scale": {"n_list": [344, 1000, 2000, 4000], "seeds": [300, 301, 302], "max_epochs": 2250,
              "checkpoints": [100, 250, 500, 1000, 1500, 2250], "eval_every": 25, "patience": 10,
              "heldout_min_epochs": 100, "n_val_frauds_ieee": 1000, "ulb_val_share": 0.20,
              "rule_window": 150, "rule_min_epochs": 800},
    "ieee": {"max_majority_train": 30000, "max_minority_train": 4000, "missing_threshold": 0.5,
             "ratios": [0.5, 1.0, 2.0, 5.0], "screen_seeds": [0, 1], "confirm_seeds": [100, 101, 102, 103, 104],
             "max_epochs": 1500, "min_epochs": 500},
    "temporal": {"screen_seeds": [0, 1], "confirm_seeds": [400, 401, 402, 403, 404]},
    "max_total_runtime_hours": 11.0,
}

CONFIG = copy.deepcopy(BASE_CONFIG)
if RUN_MODE == "quick_test":
    CONFIG.update({"augmentation_ratios": [0.5, 2.0], "screen_seeds": [0], "confirm_seeds": [100, 101, 102],
                   "max_total_runtime_hours": 1.5})
    CONFIG["diffusion"].update({"timesteps": 50, "max_epochs": 120, "min_epochs": 40, "hidden_dim": 64, "t_dim": 32})
    CONFIG["ctgan"].update({"epochs": 5})
    CONFIG["resampling"].update({"maj_ref_size": 3000})
    CONFIG["classifiers"].update({"rf_n_estimators": 20, "xgb_n_estimators": 20, "mlp_epochs": 3})
    CONFIG["mia"].update({"fixed_ref_size": 500, "n_boot": 50, "n_perm": 200})
    CONFIG["dp"].update({"noise_multipliers": [1.0, 4.0], "epochs": 5, "seeds": [200, 201]})
    CONFIG["scale"].update({"n_list": [100, 300], "seeds": [300, 301], "max_epochs": 100,
                            "checkpoints": [25, 50, 100], "eval_every": 10, "patience": 3,
                            "heldout_min_epochs": 10, "n_val_frauds_ieee": 100, "rule_window": 20,
                            "rule_min_epochs": 40})
    CONFIG["ieee"].update({"max_majority_train": 3000, "max_minority_train": 400, "ratios": [1.0, 2.0],
                           "screen_seeds": [0], "confirm_seeds": [100, 101], "max_epochs": 60, "min_epochs": 20})
    CONFIG["temporal"].update({"screen_seeds": [0], "confirm_seeds": [400, 401]})
print("RUN_MODE =", RUN_MODE)
print(json.dumps(CONFIG, indent=1, default=str))

# ------------------------------------------------------------------
# Output folders (Kaggle or local)
# ------------------------------------------------------------------
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./working")
RESULTS_DIR = os.path.join(WORK_DIR, "revision_results")
TABLES_DIR = os.path.join(RESULTS_DIR, "tables")
FIGURES_DIR = os.path.join(WORK_DIR, "revision_figures")
for d in (RESULTS_DIR, TABLES_DIR, FIGURES_DIR):
    os.makedirs(d, exist_ok=True)
with open(os.path.join(RESULTS_DIR, "config_used.json"), "w") as f:
    json.dump({"RUN_MODE": RUN_MODE, "CONFIG": CONFIG}, f, indent=2, default=str)
print("Results ->", RESULTS_DIR)

Using device: cuda
RUN_MODE = full_paper
{
 "seed": 42,
 "test_size": 0.3,
 "val_size": 0.2,
 "augmentation_ratios": [
  0.5,
  1.0,
  2.0,
  5.0,
  10.0
 ],
 "screen_seeds": [
  0,
  1,
  2
 ],
 "confirm_seeds": [
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109
 ],
 "diffusion": {
  "timesteps": 200,
  "beta_start": 0.0001,
  "beta_end": 0.02,
  "hidden_dim": 256,
  "t_dim": 128,
  "n_blocks": 4,
  "lr": 0.001,
  "batch_size": 64,
  "ema_decay": 0.995,
  "max_epochs": 3000,
  "min_epochs": 800,
  "rel_tol": 0.02,
  "loss_repeats": 20
 },
 "ctgan": {
  "epochs": 100,
  "batch_size": 500,
  "embedding_dim": 128,
  "generator_dim": [
   256,
   256
  ],
  "discriminator_dim": [
   256,
   256
  ],
  "generator_lr": 0.0002,
  "discriminator_lr": 0.0002,
  "generator_decay": 1e-06,
  "discriminator_decay": 1e-06,
  "discriminator_steps": 1,
  "pac": 10
 },
 "resampling": {
  "k_neighbors": 5,
  "maj_ref_size": 20000
 },
 "classifiers": {
  "lr_max_iter": 2000,
  "lr_C"

In [3]:
# ============================================================
# 2. Reproducibility, timing, budget guard, persistence helpers
# ============================================================
SEED = CONFIG["seed"]

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
_RUN_START = time.time()
_STAGE_LOG = []

def stage_timer(label):
    t0 = time.time()
    def _done():
        el = time.time() - t0
        _STAGE_LOG.append((label, el))
        print(f"[TIMER] {label}: {el/60:.1f} min (cumulative {(time.time()-_RUN_START)/60:.1f} min)")
        _save_runtime()
    return _done

def _save_runtime():
    path = os.path.join(RESULTS_DIR, "runtime_log.json")
    old = []
    if os.path.exists(path):
        try:
            old = json.load(open(path))["stages"]
        except Exception:
            old = []
    labels = {s["label"] for s in old}
    for l, e in _STAGE_LOG:
        if l not in labels:
            old.append({"label": l, "seconds": e, "run_mode": RUN_MODE, "device": str(DEVICE)})
    json.dump({"stages": old}, open(path, "w"), indent=2)

def time_budget_exceeded(frac=1.0):
    return (time.time() - _RUN_START) > frac * CONFIG["max_total_runtime_hours"] * 3600

def budget_status():
    return f"{(time.time()-_RUN_START)/3600:.2f} h elapsed / {CONFIG['max_total_runtime_hours']} h budget"

def rpath(name):
    return os.path.join(RESULTS_DIR, name)

def find_result(name):
    """Look for a result file in this session first, then in any attached input dataset (multi-session runs)."""
    p = rpath(name)
    if os.path.exists(p):
        return p
    hits = glob.glob(os.path.join("/kaggle/input", "**", "revision_results", name), recursive=True)
    return hits[0] if hits else None

def append_rows(name, rows):
    if not rows:
        return
    p = rpath(name)
    df = pd.DataFrame(rows)
    df.to_csv(p, mode="a", header=not os.path.exists(p), index=False)

def load_csv(name):
    p = find_result(name)
    return pd.read_csv(p) if p else None

def done_keys(name, cols):
    df = load_csv(name) if os.path.exists(rpath(name)) else None
    if df is None or len(df) == 0:
        return set()
    return set(map(tuple, df[cols].astype(str).values.tolist()))

In [4]:
# ============================================================
# 3. Statistics helpers: AUC CIs (bootstrap, DeLong), chance band, paired tests, Holm
# ============================================================
def _compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x)
    T = np.zeros(N, dtype=float); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float); T2[J] = T
    return T2

def delong_auc_ci(scores_pos, scores_neg, alpha=0.05):
    """AUC with DeLong variance (Sun & Xu 2014 fast algorithm)."""
    m, n = len(scores_pos), len(scores_neg)
    tx, ty = _compute_midrank(scores_pos), _compute_midrank(scores_neg)
    tz = _compute_midrank(np.concatenate([scores_pos, scores_neg]))
    auc = tz[:m].sum() / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:m] - tx) / n
    v10 = 1.0 - (tz[m:] - ty) / m
    var = np.var(v01, ddof=1) / m + np.var(v10, ddof=1) / n
    se = math.sqrt(max(var, 1e-12))
    z = stats.norm.ppf(1 - alpha / 2)
    return float(auc), float(max(0.0, auc - z * se)), float(min(1.0, auc + z * se)), float(se)

def auc_from_scores(scores_pos, scores_neg):
    y = np.r_[np.ones(len(scores_pos)), np.zeros(len(scores_neg))]
    return roc_auc_score(y, np.r_[scores_pos, scores_neg])

def chance_band(n_pos, n_neg, n_perm, seed=0):
    """Central 95% range of the AUC of an attack that has no information (label permutation)."""
    rng = np.random.RandomState(seed)
    y = np.r_[np.ones(n_pos), np.zeros(n_neg)]
    s = rng.rand(n_pos + n_neg)
    aucs = [roc_auc_score(rng.permutation(y), s) for _ in range(n_perm)]
    return float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))

def seed_mean_auc_bootstrap(score_list_pos, score_list_neg, n_boot, seed=0):
    """Mean AUC over generator seeds, with a percentile CI from resampling members and non-members.
    score_list_*: list over seeds of arrays aligned to the SAME member / non-member records."""
    rng = np.random.RandomState(seed)
    m, n = len(score_list_pos[0]), len(score_list_neg[0])
    point = np.mean([auc_from_scores(p, q) for p, q in zip(score_list_pos, score_list_neg)])
    boots = []
    for _ in range(n_boot):
        ip, iq = rng.randint(0, m, m), rng.randint(0, n, n)
        boots.append(np.mean([auc_from_scores(p[ip], q[iq]) for p, q in zip(score_list_pos, score_list_neg)]))
    return float(point), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

def holm(pvals):
    p = np.asarray(pvals, dtype=float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        adj[idx] = min(1.0, running)
    return adj

def paired_test(a, b, n_boot=5000, seed=0):
    """Wilcoxon signed-rank test and paired-bootstrap 95% CI of mean(a - b)."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    d = a - b
    try:
        if np.allclose(d, 0):
            W, p = float("nan"), 1.0
        else:
            W, p = stats.wilcoxon(a, b)
    except Exception:
        W, p = float("nan"), float("nan")
    rng = np.random.RandomState(seed)
    bs = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(n_boot)]
    return {"mean_diff": float(d.mean()), "ci_lo": float(np.percentile(bs, 2.5)),
            "ci_hi": float(np.percentile(bs, 97.5)), "W": float(W), "p": float(p), "n": int(len(d))}

## 1. Generators, classifiers and attacks

The diffusion architecture, noise schedule, optimiser and early-stopping rule are unchanged from the submitted
version. What changed: every generator (diffusion, CTGAN and the three resamplers) is now re-fitted for every seed,
so seed-to-seed variation includes generator randomness, and the loss attack uses a fixed random stream so that
members and non-members are scored with identical (t, noise) draws.

In [5]:
# ============================================================
# 4. Diffusion model (Gaussian DDPM, residual MLP denoiser) and training
# ============================================================
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return F.pad(emb, (0, 1)) if self.dim % 2 == 1 else emb

class ResBlock(nn.Module):
    """LayerNorm -> SiLU -> Linear (+ time projection) -> LayerNorm -> SiLU -> Linear, with a skip connection."""
    def __init__(self, dim, t_dim):
        super().__init__()
        self.norm1, self.fc1 = nn.LayerNorm(dim), nn.Linear(dim, dim)
        self.t_proj = nn.Linear(t_dim, dim)
        self.norm2, self.fc2 = nn.LayerNorm(dim), nn.Linear(dim, dim)
        self.act = nn.SiLU()
    def forward(self, x, t_emb):
        h = self.fc1(self.act(self.norm1(x))) + self.t_proj(t_emb)
        return x + self.fc2(self.act(self.norm2(h)))

class TabularDenoiser(nn.Module):
    def __init__(self, n_features, hidden_dim=256, t_dim=128, n_blocks=4):
        super().__init__()
        self.time_embed = nn.Sequential(SinusoidalTimeEmbedding(t_dim), nn.Linear(t_dim, t_dim), nn.SiLU(),
                                        nn.Linear(t_dim, t_dim))
        self.in_proj = nn.Linear(n_features, hidden_dim)
        self.blocks = nn.ModuleList([ResBlock(hidden_dim, t_dim) for _ in range(n_blocks)])
        self.out_proj = nn.Linear(hidden_dim, n_features)
    def forward(self, x, t):
        t_emb = self.time_embed(t); h = self.in_proj(x)
        for b in self.blocks:
            h = b(h, t_emb)
        return self.out_proj(h)

class DPResBlock(nn.Module):
    """Normalisation-free residual block used by the lightweight DP denoiser."""
    def __init__(self, dim, t_dim):
        super().__init__()
        self.fc1, self.t_proj, self.fc2, self.act = nn.Linear(dim, dim), nn.Linear(t_dim, dim), nn.Linear(dim, dim), nn.SiLU()
    def forward(self, x, t_emb):
        return x + self.fc2(self.act(self.fc1(x) + self.t_proj(t_emb)))

class DPTabularDenoiser(nn.Module):
    def __init__(self, n_features, hidden_dim=64, t_dim=32, n_blocks=2):
        super().__init__()
        self.time_embed = nn.Sequential(SinusoidalTimeEmbedding(t_dim), nn.Linear(t_dim, t_dim), nn.SiLU(),
                                        nn.Linear(t_dim, t_dim))
        self.in_proj = nn.Linear(n_features, hidden_dim)
        self.blocks = nn.ModuleList([DPResBlock(hidden_dim, t_dim) for _ in range(n_blocks)])
        self.out_proj = nn.Linear(hidden_dim, n_features)
    def forward(self, x, t):
        t_emb = self.time_embed(t); h = self.in_proj(x)
        for b in self.blocks:
            h = b(h, t_emb)
        return self.out_proj(h)

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.clone().detach() for k, v in model.state_dict().items()}
    def update(self, model):
        for k, v in model.state_dict().items():
            self.shadow[k] = self.decay * self.shadow[k] + (1 - self.decay) * v.detach()
    def copy_to(self, model):
        model.load_state_dict(self.shadow)

class GaussianDiffusion:
    def __init__(self, model, n_features, timesteps, beta_start=1e-4, beta_end=0.02, device=DEVICE):
        self.model, self.T, self.device, self.n_features = model, timesteps, device, n_features
        self.betas = torch.linspace(beta_start, beta_end, timesteps, device=device)
        self.alphas = 1.0 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)

    def q_sample(self, x0, t, noise):
        ab = self.alpha_bars[t].unsqueeze(1)
        return torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * noise

    def p_losses(self, x0):
        t = torch.randint(0, self.T, (x0.shape[0],), device=x0.device)
        noise = torch.randn_like(x0)
        return F.mse_loss(self.model(self.q_sample(x0, t, noise), t), noise)

    @torch.no_grad()
    def sample(self, n_samples, seed=None, batch=20000):
        g = torch.Generator(device=self.device)
        g.manual_seed(int(seed) if seed is not None else int(torch.randint(0, 2**31 - 1, (1,)).item()))
        out = []
        for start in range(0, n_samples, batch):
            nb = min(batch, n_samples - start)
            x = torch.randn(nb, self.n_features, device=self.device, generator=g)
            for i in reversed(range(self.T)):
                t = torch.full((nb,), i, device=self.device, dtype=torch.long)
                eps = self.model(x, t)
                mean = (x - self.betas[i] / torch.sqrt(1 - self.alpha_bars[i]) * eps) / torch.sqrt(self.alphas[i])
                if i > 0:
                    x = mean + torch.sqrt(self.betas[i]) * torch.randn(x.shape, device=self.device, generator=g)
                else:
                    x = mean
            out.append(x.cpu().numpy())
        return np.vstack(out) if out else np.zeros((0, self.n_features))

    @torch.no_grad()
    def per_sample_loss(self, x0_np, n_repeats=20, seed=12345, batch=8192):
        """Average noise-prediction loss per record over n_repeats (t, eps) draws. The same seeded stream is used
        for every call, so members and non-members are scored with the same kind of draws."""
        x0_all = torch.tensor(np.asarray(x0_np), dtype=torch.float32, device=self.device)
        g = torch.Generator(device=self.device); g.manual_seed(seed)
        losses = torch.zeros(x0_all.shape[0], device=self.device)
        for _ in range(n_repeats):
            for s in range(0, x0_all.shape[0], batch):
                x0 = x0_all[s:s + batch]
                t = torch.randint(0, self.T, (x0.shape[0],), device=self.device, generator=g)
                noise = torch.randn(x0.shape, device=self.device, generator=g)
                pred = self.model(self.q_sample(x0, t, noise), t)
                losses[s:s + batch] += ((pred - noise) ** 2).mean(dim=1)
        return (losses / n_repeats).cpu().numpy()


def build_denoiser(n_features, dcfg, kind="primary"):
    if kind == "primary":
        return TabularDenoiser(n_features, dcfg["hidden_dim"], dcfg["t_dim"], dcfg["n_blocks"]).to(DEVICE)
    dp = CONFIG["dp"]
    return DPTabularDenoiser(n_features, dp["light_hidden_dim"], dp["light_t_dim"], dp["light_n_blocks"]).to(DEVICE)


def train_diffusion(X_min, seed, max_epochs=None, min_epochs=None, dcfg=None, label=""):
    """Unchanged training procedure: Adam, EMA of weights, stop when the relative change of the mean training loss
    between two consecutive windows of w = max(20, max_epochs // 20) epochs falls below rel_tol (after min_epochs)."""
    dcfg = dcfg or CONFIG["diffusion"]
    max_epochs = max_epochs or dcfg["max_epochs"]; min_epochs = min_epochs or dcfg["min_epochs"]
    set_seed(seed)
    n_features = X_min.shape[1]
    model = build_denoiser(n_features, dcfg)
    diff = GaussianDiffusion(model, n_features, dcfg["timesteps"], dcfg["beta_start"], dcfg["beta_end"])
    opt = torch.optim.Adam(model.parameters(), lr=dcfg["lr"])
    ema = EMA(model, dcfg["ema_decay"])
    X_t = torch.tensor(X_min, dtype=torch.float32, device=DEVICE)
    dl = DataLoader(TensorDataset(X_t), batch_size=min(dcfg["batch_size"], len(X_t)), shuffle=True,
                    generator=torch.Generator().manual_seed(seed))
    w = max(20, max_epochs // 20)
    losses, stop_epoch, converged = [], max_epochs, False
    model.train()
    for epoch in range(max_epochs):
        tot = 0.0
        for (xb,) in dl:
            opt.zero_grad(); loss = diff.p_losses(xb); loss.backward(); opt.step(); ema.update(model)
            tot += loss.item() * len(xb)
        losses.append(tot / len(X_t))
        e = epoch + 1
        if e >= min_epochs and e >= 2 * w and e % w == 0:
            a, b = np.mean(losses[-w:]), np.mean(losses[-2 * w:-w])
            if abs(a - b) / (abs(b) + 1e-8) < dcfg["rel_tol"]:
                converged, stop_epoch = True, e
                break
    ema.copy_to(model); model.eval()
    print(f"  [diffusion {label} seed={seed}] n={len(X_min)} stopped at epoch {stop_epoch} "
          f"({'converged' if converged else 'epoch cap reached'})")
    return diff, {"losses": losses, "stop_epoch": stop_epoch, "converged": converged, "window": w}

In [6]:
# ============================================================
# 5. Baseline generators (ROS, SMOTE, ADASYN, CTGAN) and a common generator bundle
# ============================================================
def generate_resampled(method, X_min, X_maj_ref, n_synth, seed):
    """Synthetic minority records from imbalanced-learn resamplers, using a real majority-class reference so that
    ADASYN's density weights are defined on genuine local structure."""
    X_scaf = np.vstack([X_maj_ref, X_min]); y_scaf = np.r_[np.zeros(len(X_maj_ref)), np.ones(len(X_min))].astype(int)
    strategy = {1: len(X_min) + n_synth}
    k = max(1, min(CONFIG["resampling"]["k_neighbors"], len(X_min) - 1))
    def _run(m):
        if m == "ros":
            s = RandomOverSampler(sampling_strategy=strategy, random_state=seed)
        elif m == "smote":
            s = SMOTE(sampling_strategy=strategy, random_state=seed, k_neighbors=k)
        elif m == "adasyn":
            s = ADASYN(sampling_strategy=strategy, random_state=seed, n_neighbors=k)
        Xr, yr = s.fit_resample(X_scaf, y_scaf)
        return Xr[yr == 1][len(X_min):]
    try:
        Xs = _run(method)
    except Exception as e:
        print(f"[warn] {method} failed ({e}); falling back to SMOTE"); Xs = _run("smote")
    rng = np.random.RandomState(seed)
    if len(Xs) < n_synth:   # ADASYN can return slightly fewer points than requested
        Xs = np.vstack([Xs, Xs[rng.randint(0, len(Xs), n_synth - len(Xs))]])
    return Xs[:n_synth]


def fit_ctgan(X_min, seed, feature_names):
    from ctgan import CTGAN
    c = CONFIG["ctgan"]
    set_seed(seed)
    model = CTGAN(epochs=c["epochs"], batch_size=c["batch_size"], embedding_dim=c["embedding_dim"],
                  generator_dim=c["generator_dim"], discriminator_dim=c["discriminator_dim"],
                  generator_lr=c["generator_lr"], discriminator_lr=c["discriminator_lr"],
                  generator_decay=c["generator_decay"], discriminator_decay=c["discriminator_decay"],
                  discriminator_steps=c["discriminator_steps"], pac=c["pac"], verbose=False,
                  cuda=torch.cuda.is_available())
    try:
        model.set_random_state(seed)
    except Exception:
        pass
    model.fit(pd.DataFrame(X_min, columns=feature_names))
    return model


class GeneratorBundle:
    """All five augmentation methods fitted on one minority training set for one seed."""
    def __init__(self, X_min, X_maj_ref, seed, feature_names, dcfg=None, max_epochs=None, min_epochs=None,
                 use_ctgan=True, label=""):
        self.X_min, self.X_maj_ref, self.seed, self.cols = X_min, X_maj_ref, seed, feature_names
        self.diffusion, self.diff_info = train_diffusion(X_min, seed, max_epochs, min_epochs, dcfg, label)
        self.ctgan = None
        if use_ctgan and HAS_CTGAN:
            try:
                self.ctgan = fit_ctgan(X_min, seed, feature_names)
            except Exception as e:
                print(f"[warn] CTGAN failed for seed {seed}: {e}")
        self._cache = {}

    @property
    def methods(self):
        return ["ros", "smote", "adasyn", "diffusion"] + (["ctgan"] if self.ctgan is not None else [])

    def sample(self, method, n, tag=0):
        key = (method, n, tag)
        if key in self._cache:
            return self._cache[key]
        s = self.seed * 1000 + tag
        if method == "diffusion":
            X = self.diffusion.sample(n, seed=s)
        elif method == "ctgan":
            try:
                self.ctgan.set_random_state(s)
            except Exception:
                set_seed(s)
            X = self.ctgan.sample(n)[self.cols].values.astype(np.float32)
        else:
            X = generate_resampled(method, self.X_min, self.X_maj_ref, n, s)
        self._cache[key] = X
        return X

In [7]:
# ============================================================
# 6. Classifiers, metrics and threshold selection
# ============================================================
class TorchMLPClassifier:
    """Two-hidden-layer MLP trained with class-weighted BCE (pos_weight = n_neg / n_pos) on the GPU."""
    def __init__(self, hidden=(64, 32), epochs=30, batch_size=2048, lr=1e-3, random_state=0):
        self.hidden, self.epochs, self.batch_size, self.lr, self.random_state = hidden, epochs, batch_size, lr, random_state
    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        layers, d = [], X.shape[1]
        for h in self.hidden:
            layers += [nn.Linear(d, h), nn.ReLU()]; d = h
        self.model = nn.Sequential(*layers, nn.Linear(d, 1)).to(DEVICE)
        opt = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        n_pos = max(1, int(y.sum())); n_neg = max(1, len(y) - n_pos)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([n_neg / n_pos], device=DEVICE))
        Xt = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        yt = torch.tensor(y, dtype=torch.float32, device=DEVICE).unsqueeze(1)
        dl = DataLoader(TensorDataset(Xt, yt), batch_size=min(self.batch_size, len(y)), shuffle=True,
                        generator=torch.Generator().manual_seed(self.random_state))
        self.model.train()
        for _ in range(self.epochs):
            for xb, yb in dl:
                opt.zero_grad(); loss_fn(self.model(xb), yb).backward(); opt.step()
        self.model.eval(); return self
    @torch.no_grad()
    def predict_proba(self, X):
        p = torch.sigmoid(self.model(torch.tensor(X, dtype=torch.float32, device=DEVICE))).cpu().numpy().ravel()
        return np.column_stack([1 - p, p])

CLASSIFIER_NAMES = ["LogisticRegression", "RandomForest", "XGBoost", "MLP"] if HAS_XGB else \
                   ["LogisticRegression", "RandomForest", "MLP"]

def make_classifier(name, seed):
    c = CONFIG["classifiers"]
    if name == "LogisticRegression":
        return LogisticRegression(max_iter=c["lr_max_iter"], C=c["lr_C"], random_state=seed)
    if name == "RandomForest":
        return RandomForestClassifier(n_estimators=c["rf_n_estimators"], max_depth=c["rf_max_depth"],
                                      min_samples_leaf=c["rf_min_samples_leaf"], n_jobs=-1, random_state=seed)
    if name == "XGBoost":
        kw = dict(n_estimators=c["xgb_n_estimators"], max_depth=c["xgb_max_depth"], learning_rate=c["xgb_learning_rate"],
                  subsample=c["xgb_subsample"], colsample_bytree=c["xgb_colsample_bytree"], eval_metric="logloss",
                  random_state=seed, n_jobs=-1, tree_method="hist")
        if torch.cuda.is_available() and int(xgboost.__version__.split(".")[0]) >= 2:
            kw["device"] = "cuda"
        return XGBClassifier(**kw)
    if name == "MLP":
        return TorchMLPClassifier(c["mlp_hidden"], c["mlp_epochs"], c["mlp_batch_size"], c["mlp_lr"], seed)
    raise ValueError(name)

def proba(clf, X):
    return clf.predict_proba(X)[:, 1]

def metrics_at(y, p, thr):
    yp = (p >= thr).astype(int)
    return {"precision": precision_score(y, yp, zero_division=0), "recall": recall_score(y, yp, zero_division=0),
            "f1": f1_score(y, yp, zero_division=0), "mcc": matthews_corrcoef(y, yp)}

def full_metrics(y, p, thr_tuned=None):
    out = {f"{k}": v for k, v in metrics_at(y, p, CONFIG["classifiers"]["decision_threshold"]).items()}
    out["auprc"] = average_precision_score(y, p)
    out["rocauc"] = roc_auc_score(y, p)
    if thr_tuned is not None:
        mt = metrics_at(y, p, thr_tuned)
        out.update({"f1_tuned": mt["f1"], "mcc_tuned": mt["mcc"], "thr_tuned": thr_tuned})
    return out

def best_f1_threshold(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12)
    i = int(np.nanargmax(f1))
    return float(thr[i]), float(f1[i])

def train_set(X_maj, X_min, X_syn=None):
    parts = [X_maj, X_min] + ([X_syn] if X_syn is not None and len(X_syn) else [])
    X = np.vstack(parts).astype(np.float32)
    y = np.r_[np.zeros(len(X_maj)), np.ones(len(X) - len(X_maj))].astype(int)
    return X, y

In [8]:
# ============================================================
# 7. Membership-inference attacks
# ============================================================
def nn_attack_scores(query, synthetic, k):
    """Generator-agnostic attack: negated mean distance to the k nearest released synthetic records.
    Needs only the released synthetic set, so it applies to every augmentation method."""
    k = min(k, len(synthetic))
    d, _ = NearestNeighbors(n_neighbors=k).fit(synthetic).kneighbors(query)
    return -d.mean(axis=1), d[:, 0]

def exact_copy_share(X_mem, synthetic, tol=1e-6):
    _, d1 = nn_attack_scores(X_mem, synthetic, 1)
    return float(np.mean(d1 < tol))

def privacy_block(bundle, X_mem, X_non, released_ratio_map, n_ref, dataset, seed, split="random"):
    """Runs every attack for one generator seed. Returns (summary rows, per-record score dict)."""
    rows, scores = [], {}
    m, n = len(X_mem), len(X_non)
    # white-box loss attack on the diffusion model
    lm = bundle.diffusion.per_sample_loss(X_mem, CONFIG["diffusion"]["loss_repeats"])
    ln = bundle.diffusion.per_sample_loss(X_non, CONFIG["diffusion"]["loss_repeats"])
    auc, lo, hi, se = delong_auc_ci(-lm, -ln)
    rows.append({"dataset": dataset, "split": split, "seed": seed, "method": "diffusion", "attack": "loss",
                 "release": "generator", "k": np.nan, "n_synth": np.nan, "auc": auc, "delong_lo": lo,
                 "delong_hi": hi, "copy_share": np.nan})
    scores[("diffusion", "loss", "generator", 0)] = (-lm, -ln)
    # black-box nearest-neighbour attack on the released synthetic set, for every method
    for method in bundle.methods:
        for release, n_syn in [("released", int(round(released_ratio_map.get(method, 1.0) * len(bundle.X_min)))),
                               ("fixed", n_ref)]:
            S = bundle.sample(method, n_syn, tag=7 if release == "fixed" else 1)
            copy = exact_copy_share(X_mem, S)
            for k in CONFIG["mia"]["nn_k"]:
                sm, _ = nn_attack_scores(X_mem, S, k); sn, _ = nn_attack_scores(X_non, S, k)
                auc, lo, hi, se = delong_auc_ci(sm, sn)
                rows.append({"dataset": dataset, "split": split, "seed": seed, "method": method, "attack": "nn",
                             "release": release, "k": k, "n_synth": n_syn, "auc": auc, "delong_lo": lo,
                             "delong_hi": hi, "copy_share": copy})
                scores[(method, "nn", release, k)] = (sm, sn)
    return rows, scores, (lm, ln)

def save_scores(name, seed, scores):
    flat = {}
    for (method, attack, release, k), (a, b) in scores.items():
        key = f"{method}|{attack}|{release}|{k}"
        flat[key + "|mem"] = a; flat[key + "|non"] = b
    np.savez_compressed(rpath(f"{name}_seed{seed}.npz"), **flat)

## 2. Data

ULB: 30 features (`Time` and `Amount` in raw form, `V1`-`V28` PCA-transformed by the data provider), all
standardised with statistics from the training partition only. IEEE-CIS: numeric columns with at most 50%
missing values, a stratified working subsample, and median imputation computed on the training partition only
(the submitted version imputed with medians of the full table).

In [9]:
# ============================================================
# 8. Load ULB and IEEE-CIS
# ============================================================
def find_dataset(filename, roots):
    for root in roots:
        if root and os.path.isdir(root):
            hits = glob.glob(os.path.join(root, "**", filename), recursive=True)
            if hits:
                return hits[0]
    return None

ULB_ROOTS = (os.environ.get("ULB_DIR"), "/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud",
             "/kaggle/input/creditcardfraud", "/kaggle/input", "./data")
IEEE_ROOTS = (os.environ.get("IEEE_DIR"), "/kaggle/input/competitions/ieee-fraud-detection",
              "/kaggle/input/ieee-fraud-detection", "/kaggle/input", "./data")

ULB_PATH = find_dataset("creditcard.csv", ULB_ROOTS)
if ULB_PATH is None:
    raise FileNotFoundError("creditcard.csv not found: attach mlg-ulb/creditcardfraud")
ulb = pd.read_csv(ULB_PATH)
ULB_COLS = [c for c in ulb.columns if c != "Class"]
X_ulb_all = ulb[ULB_COLS].values.astype(np.float64)
y_ulb_all = ulb["Class"].values.astype(int)
print(f"ULB: {len(ulb):,} transactions, {int(y_ulb_all.sum())} fraud ({y_ulb_all.mean():.4%}), {len(ULB_COLS)} features")

# random stratified split, identical to the submitted version
ulb_tr_idx, ulb_te_idx = train_test_split(np.arange(len(y_ulb_all)), test_size=CONFIG["test_size"],
                                          stratify=y_ulb_all, random_state=SEED)
DATASET_STATS = {"ULB_n_total": int(len(ulb)), "ULB_n_fraud": int(y_ulb_all.sum()), "ULB_n_features": len(ULB_COLS),
                 "ULB_train_fraud": int(y_ulb_all[ulb_tr_idx].sum()), "ULB_test_fraud": int(y_ulb_all[ulb_te_idx].sum()),
                 "ULB_train_legit": int((y_ulb_all[ulb_tr_idx] == 0).sum()),
                 "ULB_test_legit": int((y_ulb_all[ulb_te_idx] == 0).sum())}

IEEE_AVAILABLE = False
IEEE_PATH = find_dataset("train_transaction.csv", IEEE_ROOTS) if (PARTS["D_scale"] or PARTS["E_ieee"]) else None
if IEEE_PATH is not None:
    IEEE_AVAILABLE = True
    ieee = pd.read_csv(IEEE_PATH)
    id_path = find_dataset("train_identity.csv", IEEE_ROOTS)
    if id_path is not None:
        ieee = ieee.merge(pd.read_csv(id_path), on="TransactionID", how="left")
    y_ieee_full = ieee["isFraud"].values.astype(int)
    num_cols = ieee.drop(columns=["isFraud", "TransactionID"], errors="ignore").select_dtypes(include=[np.number]).columns
    miss = ieee[num_cols].isna().mean()
    IEEE_COLS = miss[miss <= CONFIG["ieee"]["missing_threshold"]].index.tolist()
    X_ieee_full = ieee[IEEE_COLS].values.astype(np.float64)
    del ieee
    print(f"IEEE-CIS: {len(y_ieee_full):,} rows, {int(y_ieee_full.sum()):,} fraud, {len(IEEE_COLS)} numeric columns kept")

    # stratified working subsample and split, identical to the submitted version
    rng = np.random.RandomState(SEED)
    maj_all, min_all = np.where(y_ieee_full == 0)[0], np.where(y_ieee_full == 1)[0]
    maj_keep = rng.choice(maj_all, size=min(len(maj_all), CONFIG["ieee"]["max_majority_train"] * 2), replace=False)
    min_keep = rng.choice(min_all, size=min(len(min_all), CONFIG["ieee"]["max_minority_train"] * 2), replace=False)
    keep = np.concatenate([maj_keep, min_keep]); rng.shuffle(keep)
    tr_i, te_i = train_test_split(keep, test_size=CONFIG["test_size"], stratify=y_ieee_full[keep], random_state=SEED)
    tr_maj_i = tr_i[y_ieee_full[tr_i] == 0][:CONFIG["ieee"]["max_majority_train"]]
    tr_min_i = tr_i[y_ieee_full[tr_i] == 1][:CONFIG["ieee"]["max_minority_train"]]
    ieee_tr_idx = np.concatenate([tr_maj_i, tr_min_i]); ieee_te_idx = te_i
    # extra frauds never used anywhere else: validation set for held-out-loss early stopping (Part D)
    unused_min = np.setdiff1d(min_all, min_keep)
    ieee_es_idx = np.random.RandomState(SEED + 1).choice(unused_min, size=min(len(unused_min),
                                                         CONFIG["scale"]["n_val_frauds_ieee"]), replace=False)
    # median imputation with training-partition medians only
    ieee_medians = np.nanmedian(X_ieee_full[ieee_tr_idx], axis=0)
    ieee_medians = np.where(np.isnan(ieee_medians), 0.0, ieee_medians)
    def ieee_impute(idx):
        X = X_ieee_full[idx].copy(); r, c = np.where(np.isnan(X)); X[r, c] = ieee_medians[c]; return X
    DATASET_STATS.update({"IEEE_n_total": int(len(y_ieee_full)), "IEEE_n_fraud": int(y_ieee_full.sum()),
                          "IEEE_n_features": len(IEEE_COLS), "IEEE_train_fraud": int(len(tr_min_i)),
                          "IEEE_train_legit": int(len(tr_maj_i)), "IEEE_test_fraud": int(y_ieee_full[te_i].sum()),
                          "IEEE_test_legit": int((y_ieee_full[te_i] == 0).sum()), "IEEE_es_fraud": int(len(ieee_es_idx))})
else:
    print("IEEE-CIS not found: Parts D (IEEE branch) and E will be skipped.")
json.dump(DATASET_STATS, open(rpath("dataset_stats.json"), "w"), indent=2)
print(DATASET_STATS)

ULB: 284,807 transactions, 492 fraud (0.1727%), 30 features
IEEE-CIS: 590,540 rows, 20,663 fraud, 209 numeric columns kept
{'ULB_n_total': 284807, 'ULB_n_fraud': 492, 'ULB_n_features': 30, 'ULB_train_fraud': 344, 'ULB_test_fraud': 148, 'ULB_train_legit': 199020, 'ULB_test_legit': 85295, 'IEEE_n_total': 590540, 'IEEE_n_fraud': 20663, 'IEEE_n_features': 209, 'IEEE_train_fraud': 4000, 'IEEE_train_legit': 30000, 'IEEE_test_fraud': 2400, 'IEEE_test_legit': 18000, 'IEEE_es_fraud': 1000}


## 3. The evaluation protocol (used by Parts A, E and F)

1. **Selection stage.** The training partition is split into a fit part and a validation part (stratified 80/20 by
   default; the last 20% in time for the temporal split). Every generator is fitted on the fit-part frauds only,
   every (method, ratio, classifier) is trained on fit + synthetic records and scored on the validation part.
   r*(G, c) maximises mean validation F1 over the screening seeds; the tuned threshold is the median of the
   per-seed validation-optimal thresholds at r*. The primary classifier is the one with the highest validation F1.
2. **Confirmation stage.** For each fresh seed, all generators are re-fitted on all training-partition frauds,
   each classifier is trained at its selected ratio and evaluated once on the untouched test partition.
   F1/MCC are reported at the fixed 0.5 threshold and at the validation-tuned threshold; AUPRC and ROC-AUC are
   threshold-free.
3. **Privacy block** (same seeds, same generators): loss attack on the diffusion model; nearest-neighbour attack on
   the synthetic set each method actually releases at its selected ratio (and on a fixed 5,000-record reference);
   share of training frauds copied exactly; and a classifier-level attack on the trained primary classifier.

In [10]:
# ============================================================
# 9. Generic selection + confirmation protocol
# ============================================================
def run_protocol(prefix, dataset, Xtr_raw, ytr, Xte_raw, yte, cols, ratios, screen_seeds, confirm_seeds,
                 classifiers, fit_val_split="stratified", time_order=None, diff_epochs=None,
                 do_extras=False, split_label="random"):
    """prefix: file prefix for results; time_order: array giving the time rank of each training row (temporal split)."""
    dcfg = CONFIG["diffusion"]
    max_ep, min_ep = diff_epochs if diff_epochs else (dcfg["max_epochs"], dcfg["min_epochs"])
    # ---------------- selection stage ----------------
    if fit_val_split == "stratified":
        fit_i, val_i = train_test_split(np.arange(len(ytr)), test_size=CONFIG["val_size"], stratify=ytr,
                                        random_state=SEED)
    else:   # the last val_size share of the training period is the validation period
        order = np.argsort(time_order, kind="stable"); cut = int(round(len(order) * (1 - CONFIG["val_size"])))
        fit_i, val_i = order[:cut], order[cut:]
    sc1 = StandardScaler().fit(Xtr_raw[fit_i])
    Xfit, Xval = sc1.transform(Xtr_raw[fit_i]), sc1.transform(Xtr_raw[val_i])
    yfit, yval = ytr[fit_i], ytr[val_i]
    fit_maj, fit_min = Xfit[yfit == 0], Xfit[yfit == 1]
    ref1 = fit_maj[np.random.RandomState(SEED).choice(len(fit_maj), min(len(fit_maj),
                                                     CONFIG["resampling"]["maj_ref_size"]), replace=False)]
    print(f"[{prefix}] selection stage: fit {len(yfit):,} ({int(yfit.sum())} fraud), "
          f"validation {len(yval):,} ({int(yval.sum())} fraud)")
    s1_name = f"{prefix}_stage1_validation.csv"
    done = done_keys(s1_name, ["seed"])
    _t = stage_timer(f"{prefix} selection stage")
    for seed in screen_seeds:
        if (str(seed),) in done:
            continue
        if time_budget_exceeded():
            print(f"[BUDGET] {budget_status()}: stopping {prefix} selection stage"); break
        gb = GeneratorBundle(fit_min, ref1, seed, cols, max_epochs=max_ep, min_epochs=min_ep, label=f"{prefix}-fit")
        rows = []
        for method, ratio in [("none", 0.0)] + [(m, r) for m in gb.methods for r in ratios]:
            Xs = None if method == "none" else gb.sample(method, int(round(ratio * len(fit_min))))
            Xa, ya = train_set(fit_maj, fit_min, Xs)
            for cname in classifiers:
                p = proba(make_classifier(cname, seed).fit(Xa, ya), Xval)
                thr, f1v = best_f1_threshold(yval, p)
                rows.append({"dataset": dataset, "split": split_label, "stage": "validation", "seed": seed,
                             "method": method, "ratio": ratio, "classifier": cname, **full_metrics(yval, p),
                             "thr_valopt": thr, "f1_valopt": f1v})
        append_rows(s1_name, rows)
        print(f"  [{prefix}] screening seed {seed} done ({budget_status()})")
    _t()
    s1 = load_csv(s1_name)
    agg = s1.groupby(["method", "classifier", "ratio"]).agg(f1=("f1", "mean"), thr=("thr_valopt", "median")).reset_index()
    sel_ratio, sel_thr = {}, {}
    for (m, c), g in agg.groupby(["method", "classifier"]):
        row = g.loc[g["f1"].idxmax()]
        sel_ratio[f"{m}|{c}"] = float(row["ratio"]); sel_thr[f"{m}|{c}"] = float(row["thr"])
    primary = agg.groupby("classifier")["f1"].max().idxmax()
    selection = {"ratio": sel_ratio, "threshold": sel_thr, "primary_classifier": primary,
                 "n_fit": int(len(yfit)), "n_fit_fraud": int(yfit.sum()), "n_val": int(len(yval)),
                 "n_val_fraud": int(yval.sum())}
    json.dump(selection, open(rpath(f"{prefix}_selection.json"), "w"), indent=2)
    print(f"[{prefix}] primary classifier (validation): {primary}")
    # SHAP needs a tree model: the primary classifier if it is one, otherwise XGBoost / random forest
    shap_clf = primary if primary in ("XGBoost", "RandomForest") else \
        next((c for c in ("XGBoost", "RandomForest") if c in classifiers), None)

    # ---------------- confirmation stage + privacy block ----------------
    sc2 = StandardScaler().fit(Xtr_raw)
    Xtrain, Xtest = sc2.transform(Xtr_raw), sc2.transform(Xte_raw)
    tr_maj, tr_min, te_min = Xtrain[ytr == 0], Xtrain[ytr == 1], Xtest[yte == 1]
    ref2 = tr_maj[np.random.RandomState(SEED).choice(len(tr_maj), min(len(tr_maj),
                                                     CONFIG["resampling"]["maj_ref_size"]), replace=False)]
    s2_name, p_name, c_name = f"{prefix}_stage2_test.csv", f"{prefix}_privacy.csv", f"{prefix}_classifier_mia.csv"
    done = done_keys(s2_name, ["seed"])
    _t = stage_timer(f"{prefix} confirmation stage + privacy audit")
    for seed in confirm_seeds:
        if (str(seed),) in done:
            continue
        if time_budget_exceeded():
            print(f"[BUDGET] {budget_status()}: stopping {prefix} confirmation stage"); break
        gb = GeneratorBundle(tr_min, ref2, seed, cols, max_epochs=max_ep, min_epochs=min_ep, label=f"{prefix}-train")
        rows, crow, cscores = [], [], {}
        for cname in classifiers:
            for method in ["none"] + gb.methods:
                key = f"{method}|{cname}"
                r = sel_ratio.get(key, 0.0); thr = sel_thr.get(key, 0.5)
                Xs = None if method == "none" else gb.sample(method, int(round(r * len(tr_min))))
                Xa, ya = train_set(tr_maj, tr_min, Xs)
                clf = make_classifier(cname, seed).fit(Xa, ya)
                p = proba(clf, Xtest)
                rows.append({"dataset": dataset, "split": split_label, "stage": "test", "seed": seed, "method": method,
                             "ratio": r, "classifier": cname, **full_metrics(yte, p, thr)})
                if do_extras and seed == confirm_seeds[0] and method in ("none", "diffusion") and cname == shap_clf:
                    _shap_record(clf, cname, method, r, Xtest, cols)
                if cname == primary:
                    pm, pn = proba(clf, tr_min), proba(clf, te_min)
                    auc, lo, hi, _ = delong_auc_ci(pm, pn)
                    crow.append({"dataset": dataset, "split": split_label, "seed": seed, "method": method,
                                 "classifier": cname, "ratio": r, "auc": auc, "delong_lo": lo, "delong_hi": hi})
                    cscores[(method, "classifier", "released", 0)] = (pm, pn)

        append_rows(s2_name, rows); append_rows(c_name, crow)
        released = {m: sel_ratio.get(f"{m}|{primary}", 1.0) for m in gb.methods}
        prow, pscores, (lm, ln) = privacy_block(gb, tr_min, te_min, released, CONFIG["mia"]["fixed_ref_size"],
                                                dataset, seed, split_label)
        pscores.update(cscores)
        for r_ in prow:
            r_["diff_stop_epoch"] = gb.diff_info["stop_epoch"]
        append_rows(p_name, prow); save_scores(f"{prefix}_scores", seed, pscores)
        append_rows(f"{prefix}_diffusion_loss_curves.csv",
                    [{"seed": seed, "epoch": i + 1, "loss": l} for i, l in enumerate(gb.diff_info["losses"])])
        if do_extras and seed == confirm_seeds[0]:
            _pca_record(prefix, gb, tr_maj, tr_min)
        del gb
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"  [{prefix}] confirmation seed {seed} done ({budget_status()})")
    _t()
    return selection


def _shap_record(clf, cname, method, ratio, Xtest, cols):
    if cname not in ("XGBoost", "RandomForest"):
        return
    try:
        import shap
        idx = np.random.RandomState(SEED).choice(len(Xtest), size=min(1500, len(Xtest)), replace=False)
        sv = shap.TreeExplainer(clf).shap_values(Xtest[idx])
        if isinstance(sv, list):
            sv = sv[1]
        if sv.ndim == 3:
            sv = sv[:, :, 1]
        imp = np.abs(sv).mean(axis=0)
        append_rows("ulb_shap.csv", [{"method": method, "ratio": ratio, "classifier": cname, "feature": f,
                                      "mean_abs_shap": float(v)} for f, v in zip(cols, imp)])
    except Exception as e:
        print(f"[warn] SHAP failed: {e}")


def _pca_record(prefix, gb, tr_maj, tr_min):
    idx = np.random.RandomState(SEED).choice(len(tr_maj), size=min(len(tr_maj), 5000), replace=False)
    pca = PCA(n_components=2, random_state=SEED).fit(np.vstack([tr_maj[idx], tr_min]))
    rows = []
    for g, X in [("real legitimate", tr_maj[idx]), ("real fraud", tr_min)]:
        Z = pca.transform(X); rows += [{"group": g, "method": "", "pc1": a, "pc2": b} for a, b in Z]
    for m in gb.methods:
        Z = pca.transform(gb.sample(m, len(tr_min), tag=3))
        rows += [{"group": "synthetic", "method": m, "pc1": a, "pc2": b} for a, b in Z]
    pd.DataFrame(rows).to_csv(rpath(f"{prefix}_pca_fidelity.csv"), index=False)

## Part A. ULB: utility, statistical tests and the full privacy audit

Members are the 344 training-partition frauds, non-members the 148 test-partition frauds. The white-box loss attack
targets a released generator; the nearest-neighbour attack targets a released synthetic set (and therefore applies
to ROS, SMOTE and ADASYN as well); the classifier-level attack targets a released classifier, which is always trained
on the real training frauds (Eq. 2), so generator-level DP cannot protect them at that level.

In [11]:
if PARTS["A_ulb_main"]:
    X_tr_raw, y_tr = X_ulb_all[ulb_tr_idx], y_ulb_all[ulb_tr_idx]
    X_te_raw, y_te = X_ulb_all[ulb_te_idx], y_ulb_all[ulb_te_idx]
    SEL_ULB = run_protocol("ulb", "ULB", X_tr_raw, y_tr, X_te_raw, y_te, ULB_COLS, CONFIG["augmentation_ratios"],
                           CONFIG["screen_seeds"], CONFIG["confirm_seeds"], CLASSIFIER_NAMES, do_extras=True)
else:
    SEL_ULB = json.load(open(find_result("ulb_selection.json"))) if find_result("ulb_selection.json") else None

[ulb] selection stage: fit 159,491 (275 fraud), validation 39,873 (69 fraud)
  [diffusion ulb-fit seed=0] n=275 stopped at epoch 1800 (converged)
  [ulb] screening seed 0 done (0.58 h elapsed / 11.0 h budget)
  [diffusion ulb-fit seed=1] n=275 stopped at epoch 1500 (converged)
  [ulb] screening seed 1 done (1.14 h elapsed / 11.0 h budget)
  [diffusion ulb-fit seed=2] n=275 stopped at epoch 1500 (converged)
  [ulb] screening seed 2 done (1.70 h elapsed / 11.0 h budget)
[TIMER] ulb selection stage: 101.1 min (cumulative 102.1 min)
[ulb] primary classifier (validation): XGBoost
  [diffusion ulb-train seed=100] n=344 stopped at epoch 2100 (converged)
  [ulb] confirmation seed 100 done (1.89 h elapsed / 11.0 h budget)
  [diffusion ulb-train seed=101] n=344 stopped at epoch 1950 (converged)
  [ulb] confirmation seed 101 done (2.07 h elapsed / 11.0 h budget)
  [diffusion ulb-train seed=102] n=344 stopped at epoch 1950 (converged)
  [ulb] confirmation seed 102 done (2.26 h elapsed / 11.0 h bud

## Part C. ULB: DP-SGD, both architectures, with non-DP references

For each seed, both denoisers (the primary LayerNorm architecture and the lightweight normalisation-free one) are
trained for the same number of epochs without DP and with DP-SGD at each noise multiplier. Utility is measured with
the primary classifier at the ratio selected for diffusion in Part A, over the same seeds, so every row of the DP
table is directly comparable. The undefended, early-stopped primary model and the no-augmentation baseline come
from Part A.

In [12]:
if PARTS["C_dp"] and HAS_OPACUS:
    from opacus import PrivacyEngine
    dpc, dcfg = CONFIG["dp"], CONFIG["diffusion"]
    sel = SEL_ULB or {"ratio": {}, "threshold": {}, "primary_classifier": "XGBoost" if HAS_XGB else "RandomForest"}
    prim = sel["primary_classifier"]
    r_diff = sel["ratio"].get(f"diffusion|{prim}", 0.5); thr_diff = sel["threshold"].get(f"diffusion|{prim}", 0.5)
    sc = StandardScaler().fit(X_ulb_all[ulb_tr_idx])
    Xtrain, Xtest = sc.transform(X_ulb_all[ulb_tr_idx]), sc.transform(X_ulb_all[ulb_te_idx])
    ytr_, yte_ = y_ulb_all[ulb_tr_idx], y_ulb_all[ulb_te_idx]
    tr_maj, tr_min, te_min = Xtrain[ytr_ == 0], Xtrain[ytr_ == 1], Xtest[yte_ == 1]
    nf = tr_min.shape[1]
    done = done_keys("ulb_dp.csv", ["seed", "arch", "noise_multiplier"])
    _t = stage_timer("DP-SGD sweep (both architectures, non-DP references)")
    for seed in dpc["seeds"]:
        for arch in ["light", "matched"]:
            for nm in [None] + dpc["noise_multipliers"]:
                if (str(seed), arch, str(nm if nm is not None else "none")) in done:
                    continue
                if time_budget_exceeded():
                    print(f"[BUDGET] {budget_status()}: stopping DP sweep"); break
                set_seed(seed)
                model = build_denoiser(nf, dcfg, "primary" if arch == "matched" else "light")
                diff = GaussianDiffusion(model, nf, dcfg["timesteps"], dcfg["beta_start"], dcfg["beta_end"])
                opt = torch.optim.Adam(model.parameters(), lr=dcfg["lr"])
                dl = DataLoader(TensorDataset(torch.tensor(tr_min, dtype=torch.float32)),
                                batch_size=min(dcfg["batch_size"], len(tr_min)), shuffle=True,
                                generator=torch.Generator().manual_seed(seed))
                pe = None
                if nm is not None:
                    pe = PrivacyEngine()
                    model, opt, dl = pe.make_private(module=model, optimizer=opt, data_loader=dl,
                                                     noise_multiplier=nm, max_grad_norm=dpc["max_grad_norm"])
                    diff.model = model
                model.train()
                for epoch in range(dpc["epochs"]):
                    for (xb,) in dl:
                        xb = xb.to(DEVICE)
                        if xb.shape[0] == 0:
                            continue
                        opt.zero_grad(); diff.p_losses(xb).backward(); opt.step()
                model.eval()
                eps = pe.get_epsilon(delta=dpc["delta"]) if pe is not None else float("inf")
                lm = diff.per_sample_loss(tr_min, dcfg["loss_repeats"]); ln = diff.per_sample_loss(te_min, dcfg["loss_repeats"])
                auc_loss, lo, hi, _ = delong_auc_ci(-lm, -ln)
                S = diff.sample(int(round(r_diff * len(tr_min))), seed=seed * 1000 + 1)
                sm, _ = nn_attack_scores(tr_min, S, 1); sn, _ = nn_attack_scores(te_min, S, 1)
                auc_nn = delong_auc_ci(sm, sn)[0]
                Xa, ya = train_set(tr_maj, tr_min, S)
                p = proba(make_classifier(prim, seed).fit(Xa, ya), Xtest)
                met = full_metrics(yte_, p, thr_diff)
                append_rows("ulb_dp.csv", [{"seed": seed, "arch": arch,
                                            "noise_multiplier": nm if nm is not None else "none",
                                            "epsilon": eps, "epochs": dpc["epochs"], "classifier": prim,
                                            "ratio": r_diff, "auc_loss": auc_loss, "auc_loss_lo": lo,
                                            "auc_loss_hi": hi, "auc_nn": auc_nn, **met}])
                print(f"  DP seed={seed} arch={arch} sigma={nm}: eps={eps:.2f} F1={met['f1']:.4f} "
                      f"lossAUC={auc_loss:.4f} nnAUC={auc_nn:.4f}")
                del model, diff, opt, pe
    _t()
elif PARTS["C_dp"]:
    print("Opacus unavailable: Part C skipped")

  DP seed=200 arch=light sigma=None: eps=inf F1=0.8417 lossAUC=0.5949 nnAUC=0.4893
  DP seed=200 arch=light sigma=0.5: eps=311.72 F1=0.8592 lossAUC=0.5270 nnAUC=0.4888
  DP seed=200 arch=light sigma=1.0: eps=63.96 F1=0.8448 lossAUC=0.5259 nnAUC=0.4808
  DP seed=200 arch=light sigma=2.0: eps=19.99 F1=0.8375 lossAUC=0.5247 nnAUC=0.4797
  DP seed=200 arch=light sigma=4.0: eps=7.91 F1=0.8551 lossAUC=0.5305 nnAUC=0.4768
  DP seed=200 arch=matched sigma=None: eps=inf F1=0.8487 lossAUC=0.7975 nnAUC=0.4997
  DP seed=200 arch=matched sigma=0.5: eps=311.72 F1=0.8417 lossAUC=0.5669 nnAUC=0.4917
  DP seed=200 arch=matched sigma=1.0: eps=63.96 F1=0.8417 lossAUC=0.5495 nnAUC=0.4873
  DP seed=200 arch=matched sigma=2.0: eps=19.99 F1=0.8406 lossAUC=0.5373 nnAUC=0.4824
  DP seed=200 arch=matched sigma=4.0: eps=7.91 F1=0.8507 lossAUC=0.5388 nnAUC=0.4795
  DP seed=201 arch=light sigma=None: eps=inf F1=0.8456 lossAUC=0.5969 nnAUC=0.4913
  DP seed=201 arch=light sigma=0.5: eps=311.72 F1=0.8528 lossAUC=0.52

## Part D. Does leakage track data scale or training length?

IEEE-CIS: fixed architecture, optimiser and epoch budget; only the number of training frauds n changes (random
subsets of the 4,000 training frauds; non-members are the 2,400 test frauds; a further 1,000 unused frauds serve as the
early-stopping validation set). The loss-attack AUC is recorded at fixed epochs and at three stopping points: the
submitted training-loss rule (Eq. 7), held-out-loss early stopping, and the epoch cap. ULB: the same curve with 80% of
the training frauds as members and the remaining 20% as the held-out validation set.

In [13]:
def train_diffusion_tracked(X_mem, X_non, X_val, seed, dcfg, sc):
    set_seed(seed)
    nf = X_mem.shape[1]
    model = build_denoiser(nf, dcfg); diff = GaussianDiffusion(model, nf, dcfg["timesteps"], dcfg["beta_start"], dcfg["beta_end"])
    ev_model = build_denoiser(nf, dcfg); ev = GaussianDiffusion(ev_model, nf, dcfg["timesteps"], dcfg["beta_start"], dcfg["beta_end"])
    opt = torch.optim.Adam(model.parameters(), lr=dcfg["lr"]); ema = EMA(model, dcfg["ema_decay"])
    Xt = torch.tensor(X_mem, dtype=torch.float32, device=DEVICE)
    dl = DataLoader(TensorDataset(Xt), batch_size=min(dcfg["batch_size"], len(Xt)), shuffle=True,
                    generator=torch.Generator().manual_seed(seed))
    losses, recs, curve = [], [], []
    best_val, best_state, best_ep, bad = float("inf"), None, None, 0
    ho_state, ho_ep, ho_stop = None, None, None
    rule_state, rule_ep = None, None
    w = sc["rule_window"]

    def attack(state):
        ev_model.load_state_dict(state); ev_model.eval()
        lm = ev.per_sample_loss(X_mem, dcfg["loss_repeats"]); ln = ev.per_sample_loss(X_non, dcfg["loss_repeats"])
        auc, lo, hi, _ = delong_auc_ci(-lm, -ln)
        return auc, lo, hi, float(lm.mean()), float(ln.mean())

    model.train()
    for epoch in range(sc["max_epochs"]):
        tot = 0.0
        for (xb,) in dl:
            opt.zero_grad(); loss = diff.p_losses(xb); loss.backward(); opt.step(); ema.update(model)
            tot += loss.item() * len(xb)
        losses.append(tot / len(Xt)); e = epoch + 1
        if rule_ep is None and e >= sc["rule_min_epochs"] and e >= 2 * w and e % w == 0:
            a, b = np.mean(losses[-w:]), np.mean(losses[-2 * w:-w])
            if abs(a - b) / (abs(b) + 1e-8) < dcfg["rel_tol"]:
                rule_ep, rule_state = e, {k: v.clone() for k, v in ema.shadow.items()}
        if e % sc["eval_every"] == 0:
            ev_model.load_state_dict(ema.shadow); ev_model.eval()
            v = float(ev.per_sample_loss(X_val, 5, seed=999).mean())
            curve.append({"epoch": e, "val_loss": v, "train_loss": losses[-1]})
            if v < best_val - 1e-6:
                best_val, best_ep, bad = v, e, 0
                best_state = {k: t.clone() for k, t in ema.shadow.items()}
            elif e >= sc["heldout_min_epochs"]:
                bad += 1
            if ho_stop is None and bad >= sc["patience"] and e >= sc["heldout_min_epochs"]:
                ho_stop, ho_ep, ho_state = e, best_ep, best_state
        if e in sc["checkpoints"]:
            auc, lo, hi, mm, mn = attack(ema.shadow)
            recs.append({"rule": "fixed", "epoch": e, "auc": auc, "lo": lo, "hi": hi, "loss_mem": mm, "loss_non": mn})
    if ho_state is None:
        ho_ep, ho_state, ho_stop = best_ep, best_state, None
    for rule, ep, st in [("train_loss_rule", rule_ep, rule_state), ("heldout", ho_ep, ho_state)]:
        if st is not None:
            auc, lo, hi, mm, mn = attack(st)
            recs.append({"rule": rule, "epoch": ep, "auc": auc, "lo": lo, "hi": hi, "loss_mem": mm, "loss_non": mn,
                         "stop_epoch": ho_stop if rule == "heldout" else ep})
        else:
            recs.append({"rule": rule, "epoch": np.nan, "auc": np.nan, "lo": np.nan, "hi": np.nan,
                         "loss_mem": np.nan, "loss_non": np.nan, "stop_epoch": np.nan})
    return recs, curve

if PARTS["D_scale"]:
    scfg, dcfg = CONFIG["scale"], CONFIG["diffusion"]
    done = done_keys("scale_results.csv", ["dataset", "n", "seed"])
    _t = stage_timer("Part D: data scale and training length")
    jobs = []
    # ULB: 80% of training frauds as members, 20% as held-out validation, test frauds as non-members
    sc_u = StandardScaler().fit(X_ulb_all[ulb_tr_idx])
    u_tr_min = sc_u.transform(X_ulb_all[ulb_tr_idx][y_ulb_all[ulb_tr_idx] == 1])
    u_te_min = sc_u.transform(X_ulb_all[ulb_te_idx][y_ulb_all[ulb_te_idx] == 1])
    for seed in scfg["seeds"]:
        mem_i, val_i = train_test_split(np.arange(len(u_tr_min)), test_size=scfg["ulb_val_share"], random_state=seed)
        jobs.append(("ULB", len(mem_i), seed, u_tr_min[mem_i], u_te_min, u_tr_min[val_i]))
    if IEEE_AVAILABLE:
        sc_i = StandardScaler().fit(ieee_impute(ieee_tr_idx))
        i_tr = sc_i.transform(ieee_impute(ieee_tr_idx)); i_ytr = y_ieee_full[ieee_tr_idx]
        i_te = sc_i.transform(ieee_impute(ieee_te_idx)); i_yte = y_ieee_full[ieee_te_idx]
        i_tr_min, i_te_min = i_tr[i_ytr == 1], i_te[i_yte == 1]
        i_val = sc_i.transform(ieee_impute(ieee_es_idx))
        for seed in scfg["seeds"]:
            for n in scfg["n_list"]:
                sub = np.random.RandomState(seed).choice(len(i_tr_min), size=min(n, len(i_tr_min)), replace=False)
                jobs.append(("IEEE-CIS", n, seed, i_tr_min[sub], i_te_min, i_val))
    for ds, n, seed, Xm, Xn, Xv in jobs:
        if (ds, str(n), str(seed)) in done:
            continue
        if time_budget_exceeded():
            print(f"[BUDGET] {budget_status()}: stopping Part D"); break
        t0 = time.time()
        recs, curve = train_diffusion_tracked(Xm, Xn, Xv, seed, dcfg, scfg)
        append_rows("scale_results.csv", [{"dataset": ds, "n": n, "seed": seed, "n_nonmembers": len(Xn), **r} for r in recs])
        append_rows("scale_val_curves.csv", [{"dataset": ds, "n": n, "seed": seed, **c} for c in curve])
        print(f"  Part D {ds} n={n} seed={seed}: {(time.time()-t0)/60:.1f} min; "
              + ", ".join(f"{r['rule']}@{r['epoch']}={r['auc']:.3f}" for r in recs if r["rule"] != "fixed"))
    _t()

  Part D ULB n=275 seed=300: 1.3 min; train_loss_rule@1800=0.930, heldout@175=0.758
  Part D ULB n=275 seed=301: 1.3 min; train_loss_rule@1950=0.931, heldout@225=0.775
  Part D ULB n=275 seed=302: 1.3 min; train_loss_rule@1500=0.921, heldout@150=0.729
  Part D IEEE-CIS n=344 seed=300: 1.6 min; train_loss_rule@1950=0.731, heldout@125=0.586
  Part D IEEE-CIS n=1000 seed=300: 4.2 min; train_loss_rule@1950=0.610, heldout@1250=0.603
  Part D IEEE-CIS n=2000 seed=300: 8.4 min; train_loss_rule@1800=0.561, heldout@2175=0.565
  Part D IEEE-CIS n=4000 seed=300: 16.4 min; train_loss_rule@1350=0.525, heldout@2075=0.528
  Part D IEEE-CIS n=344 seed=301: 1.6 min; train_loss_rule@nan=nan, heldout@475=0.647
  Part D IEEE-CIS n=1000 seed=301: 4.2 min; train_loss_rule@1950=0.616, heldout@1475=0.609
  Part D IEEE-CIS n=2000 seed=301: 8.3 min; train_loss_rule@1650=0.547, heldout@825=0.540
  Part D IEEE-CIS n=4000 seed=301: 16.3 min; train_loss_rule@1500=0.526, heldout@1550=0.527
  Part D IEEE-CIS n=344 se

## Part E. IEEE-CIS: same protocol as Part A (5 confirmation seeds, SDs for every metric)

In [14]:
if PARTS["E_ieee"] and IEEE_AVAILABLE:
    SEL_IEEE = run_protocol("ieee", "IEEE-CIS", ieee_impute(ieee_tr_idx), y_ieee_full[ieee_tr_idx],
                            ieee_impute(ieee_te_idx), y_ieee_full[ieee_te_idx], IEEE_COLS, CONFIG["ieee"]["ratios"],
                            CONFIG["ieee"]["screen_seeds"], CONFIG["ieee"]["confirm_seeds"], CLASSIFIER_NAMES,
                            diff_epochs=(CONFIG["ieee"]["max_epochs"], CONFIG["ieee"]["min_epochs"]))

[ieee] selection stage: fit 27,200 (3200 fraud), validation 6,800 (800 fraud)
  [diffusion ieee-fit seed=0] n=3200 stopped at epoch 900 (converged)
  [ieee] screening seed 0 done (5.75 h elapsed / 11.0 h budget)
  [diffusion ieee-fit seed=1] n=3200 stopped at epoch 750 (converged)
  [ieee] screening seed 1 done (5.99 h elapsed / 11.0 h budget)
[TIMER] ieee selection stage: 29.2 min (cumulative 359.4 min)
[ieee] primary classifier (validation): XGBoost
  [diffusion ieee-train seed=100] n=4000 stopped at epoch 750 (converged)
  [ieee] confirmation seed 100 done (6.19 h elapsed / 11.0 h budget)
  [diffusion ieee-train seed=101] n=4000 stopped at epoch 825 (converged)
  [ieee] confirmation seed 101 done (6.40 h elapsed / 11.0 h budget)
  [diffusion ieee-train seed=102] n=4000 stopped at epoch 750 (converged)
  [ieee] confirmation seed 102 done (6.60 h elapsed / 11.0 h budget)
  [diffusion ieee-train seed=103] n=4000 stopped at epoch 750 (converged)
  [ieee] confirmation seed 103 done (6.81

## Part F. ULB with a time-ordered split

The first 70% of transactions (by `Time`) form the training partition and the last 30% the test partition; within the
training partition the last 20% in time is the validation period. Only the primary classifier from Part A is used.
Non-members come from a later period than members, so the loss attack here mixes membership with temporal drift and is
reported for completeness only.

In [15]:
if PARTS["F_temporal"]:
    order = np.argsort(ulb["Time"].values, kind="stable")
    cut = int(round(len(order) * (1 - CONFIG["test_size"])))
    t_tr, t_te = order[:cut], order[cut:]
    prim = (SEL_ULB or {}).get("primary_classifier", "XGBoost" if HAS_XGB else "RandomForest")
    DATASET_STATS.update({"TEMP_train_fraud": int(y_ulb_all[t_tr].sum()), "TEMP_test_fraud": int(y_ulb_all[t_te].sum())})
    json.dump(DATASET_STATS, open(rpath("dataset_stats.json"), "w"), indent=2)
    SEL_TEMP = run_protocol("temporal", "ULB", X_ulb_all[t_tr], y_ulb_all[t_tr], X_ulb_all[t_te], y_ulb_all[t_te],
                            ULB_COLS, CONFIG["augmentation_ratios"], CONFIG["temporal"]["screen_seeds"],
                            CONFIG["temporal"]["confirm_seeds"], [prim], fit_val_split="temporal",
                            time_order=ulb["Time"].values[t_tr], split_label="temporal")

[temporal] selection stage: fit 159,492 (356 fraud), validation 39,873 (28 fraud)
  [diffusion temporal-fit seed=0] n=356 stopped at epoch 2250 (converged)
  [temporal] screening seed 0 done (7.04 h elapsed / 11.0 h budget)
  [diffusion temporal-fit seed=1] n=356 stopped at epoch 1800 (converged)
  [temporal] screening seed 1 done (7.07 h elapsed / 11.0 h budget)
[TIMER] temporal selection stage: 3.8 min (cumulative 424.1 min)
[temporal] primary classifier (validation): XGBoost
  [diffusion temporal-train seed=400] n=384 stopped at epoch 1650 (converged)
  [temporal] confirmation seed 400 done (7.09 h elapsed / 11.0 h budget)
  [diffusion temporal-train seed=401] n=384 stopped at epoch 2250 (converged)
  [temporal] confirmation seed 401 done (7.12 h elapsed / 11.0 h budget)
  [diffusion temporal-train seed=402] n=384 stopped at epoch 1950 (converged)
  [temporal] confirmation seed 402 done (7.15 h elapsed / 11.0 h budget)
  [diffusion temporal-train seed=403] n=384 stopped at epoch 210

## Part G. Tables, figures, editable chart workbook and manuscript values

Everything below reads only the CSV/NPZ/JSON files written by Parts A-F, so it can be re-run on its own (for example
in a later session with the results folder attached as a dataset). Figures use 2x2 or 2x3 panel grids so that labels
stay legible at print width, and carry no embedded "Figure N" titles (numbering belongs to the manuscript captions).

In [16]:
# ============================================================
# 10. Formatting helpers and the manuscript value store
# ============================================================
METHODS = ["none", "ros", "smote", "adasyn", "ctgan", "diffusion"]
AUG_METHODS = ["ros", "smote", "adasyn", "ctgan", "diffusion"]
LABEL = {"none": "No augmentation", "ros": "ROS", "smote": "SMOTE", "adasyn": "ADASYN", "ctgan": "CTGAN",
         "diffusion": "Diffusion (DDPM)"}
PALETTE = {"diffusion": "#0072B2", "smote": "#E69F00", "adasyn": "#009E73", "ros": "#CC79A7", "ctgan": "#D55E00",
           "none": "#333333", "member": "#0072B2", "nonmember": "#D55E00", "ULB": "#0072B2", "IEEE-CIS": "#D55E00"}
VALUES = {}

def f3(x):
    return "n/a" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{x:.3f}"

def pm(vals, d=3):
    v = np.asarray(vals, float); v = v[np.isfinite(v)]
    if len(v) == 0:
        return "n/a"
    return f"{v.mean():.{d}f} ± {v.std(ddof=1):.{d}f}" if len(v) > 1 else f"{v.mean():.{d}f}"

def fp(p):
    if p is None or not np.isfinite(p):
        return "n/a"
    return "<0.001" if p < 0.001 else f"{p:.3f}"

def fd(r):
    return f"{r['mean_diff']:+.3f} [{r['ci_lo']:+.3f}, {r['ci_hi']:+.3f}]"

def fratio(r):
    return "n/a" if r is None or not np.isfinite(r) else (f"{r:g}×" if r > 0 else "none")

def setv(key, val):
    VALUES[key] = val

def set_pub_style():
    plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.family": "serif", "font.size": 12,
                         "axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 12,
                         "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3,
                         "grid.linestyle": "--", "legend.frameon": False, "legend.fontsize": 11,
                         "xtick.labelsize": 11, "ytick.labelsize": 11, "lines.linewidth": 2.0, "lines.markersize": 6})
set_pub_style()

def save_fig(fig, name):
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIGURES_DIR, f"{name}.{ext}"), dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("saved", name)

FIG_DATA = {}   # figure name -> list of (sheet title, DataFrame, chart spec)

def load_npz_scores(prefix):
    out = {}
    for p in sorted(glob.glob(rpath(f"{prefix}_scores_seed*.npz"))) or \
             sorted(glob.glob(os.path.join("/kaggle/input", "**", "revision_results", f"{prefix}_scores_seed*.npz"),
                              recursive=True)):
        seed = int(os.path.basename(p).split("seed")[1].split(".")[0])
        z = np.load(p); out[seed] = {k: z[k] for k in z.files}
    return out

In [17]:
# ============================================================
# 11. Utility and statistical-test tables (ULB, IEEE-CIS, temporal)
# ============================================================
def utility_block(prefix, tag, sel):
    s2 = load_csv(f"{prefix}_stage2_test.csv")
    if s2 is None or sel is None:
        print(f"[{tag}] no confirmation results found"); return None
    prim = sel["primary_classifier"]
    setv(f"{tag}_PRIMARY", prim)
    d = s2[s2.classifier == prim]
    setv(f"{tag}_NSEEDS", str(d.seed.nunique()))
    rows = []
    for m in METHODS:
        g = d[d.method == m]
        if len(g) == 0:
            continue
        r = sel["ratio"].get(f"{m}|{prim}", 0.0) if m != "none" else 0.0
        setv(f"{tag}_{m}_r", fratio(r))
        for met, key in [("f1", "f1"), ("f1_tuned", "f1t"), ("auprc", "auprc"), ("mcc", "mcc"), ("rocauc", "roc"),
                         ("precision", "prec"), ("recall", "rec")]:
            if met in g:
                setv(f"{tag}_{m}_{key}", pm(g[met])); setv(f"{tag}_{m}_{key}m", f3(g[met].mean()))
        rows.append({"Method": LABEL[m], "Selected ratio r*": fratio(r), "F1 (0.5)": pm(g.f1),
                     "F1 (tuned)": pm(g.f1_tuned) if "f1_tuned" in g else "n/a", "AUPRC": pm(g.auprc),
                     "MCC (0.5)": pm(g.mcc), "ROC-AUC": pm(g.rocauc), "Seeds": g.seed.nunique()})
    tab = pd.DataFrame(rows); tab.to_csv(os.path.join(TABLES_DIR, f"{prefix}_utility_primary.csv"), index=False)
    # all classifiers (supplementary)
    allc = s2.groupby(["classifier", "method"]).agg(ratio=("ratio", "first"), f1=("f1", "mean"), f1_sd=("f1", "std"),
                                                    auprc=("auprc", "mean"), auprc_sd=("auprc", "std"),
                                                    mcc=("mcc", "mean"), mcc_sd=("mcc", "std")).reset_index()
    allc.to_csv(os.path.join(TABLES_DIR, f"{prefix}_utility_all_classifiers.csv"), index=False)
    # paired tests: diffusion against every other condition (Holm over the family), for F1 and AUPRC
    piv = {met: d.pivot_table(index="seed", columns="method", values=met) for met in ("f1", "auprc")}
    trows = []
    for met, pref in [("f1", "TST"), ("auprc", "TSTA")]:
        P = piv[met]
        others = [m for m in METHODS if m != "diffusion" and m in P.columns]
        if "diffusion" not in P.columns:
            continue
        res = {m: paired_test(P["diffusion"].values, P[m].values) for m in others}
        adj = holm([res[m]["p"] for m in others])
        for m, pa in zip(others, adj):
            r = res[m]; r["p_holm"] = pa
            setv(f"{pref}_{tag}_{m}_d", fd(r)); setv(f"{pref}_{tag}_{m}_p", fp(r["p"]))
            setv(f"{pref}_{tag}_{m}_ph", fp(pa)); setv(f"{pref}_{tag}_{m}_W", f"{r['W']:.1f}" if np.isfinite(r["W"]) else "n/a")
            trows.append({"Metric": met.upper(), "Comparison": f"Diffusion vs {LABEL[m]}", "Mean difference [95% CI]": fd(r),
                          "W": r["W"], "p": fp(r["p"]), "p (Holm)": fp(pa), "n": r["n"]})
        # secondary family: CTGAN against no augmentation and against the best resampler
        if met == "f1" and "ctgan" in P.columns:
            sec = [m for m in ["none", "smote", "ros", "adasyn"] if m in P.columns]
            res2 = {m: paired_test(P["ctgan"].values, P[m].values) for m in sec}
            adj2 = holm([res2[m]["p"] for m in sec])
            for m, pa in zip(sec, adj2):
                r = res2[m]
                setv(f"TST_{tag}_ctgan_{m}_d", fd(r)); setv(f"TST_{tag}_ctgan_{m}_ph", fp(pa))
                trows.append({"Metric": "F1", "Comparison": f"CTGAN vs {LABEL[m]}", "Mean difference [95% CI]": fd(r),
                              "W": r["W"], "p": fp(r["p"]), "p (Holm)": fp(pa), "n": r["n"]})
    tt = pd.DataFrame(trows)
    if len(tt):
        tt["W"] = tt["W"].apply(lambda w: f"{w:.1f}" if isinstance(w, (int, float)) and np.isfinite(w) else "n/a")
    tt.to_csv(os.path.join(TABLES_DIR, f"{prefix}_paired_tests.csv"), index=False)
    return d

def load_sel(prefix):
    p = find_result(f"{prefix}_selection.json")
    return json.load(open(p)) if p else None

if PARTS["G_assemble"]:
    stats_path = find_result("dataset_stats.json")
    if stats_path:
        for k, v in json.load(open(stats_path)).items():
            setv(f"DS_{k}", f"{v:,}" if isinstance(v, int) else str(v))
    SEL = {"ulb": load_sel("ulb"), "ieee": load_sel("ieee"), "temporal": load_sel("temporal")}
    D_ULB = utility_block("ulb", "ULB", SEL["ulb"])
    D_IEEE = utility_block("ieee", "IEEE", SEL["ieee"])
    D_TMP = utility_block("temporal", "TMP", SEL["temporal"])
    for pref, tag in [("ulb", "ULB"), ("ieee", "IEEE"), ("temporal", "TMP")]:
        if SEL[pref]:
            setv(f"{tag}_nfit", f"{SEL[pref]['n_fit']:,}"); setv(f"{tag}_nfitfraud", f"{SEL[pref]['n_fit_fraud']:,}")
            setv(f"{tag}_nval", f"{SEL[pref]['n_val']:,}"); setv(f"{tag}_nvalfraud", f"{SEL[pref]['n_val_fraud']:,}")

In [18]:
# ============================================================
# 12. Privacy tables (ULB, IEEE-CIS, temporal) with seed-averaged bootstrap CIs and the chance band
# ============================================================
def privacy_tables(prefix, tag):
    pr = load_csv(f"{prefix}_privacy.csv"); cm = load_csv(f"{prefix}_classifier_mia.csv")
    sc = load_npz_scores(prefix)
    if pr is None or not sc:
        print(f"[{tag}] no privacy results"); return None
    seeds = sorted(sc)
    first = sc[seeds[0]]
    n_mem = len(first["diffusion|loss|generator|0|mem"]); n_non = len(first["diffusion|loss|generator|0|non"])
    lo_c, hi_c = chance_band(n_mem, n_non, CONFIG["mia"]["n_perm"])
    setv(f"MIA_{tag}_chance", f"{lo_c:.3f}–{hi_c:.3f}"); setv(f"MIA_{tag}_nmem", f"{n_mem:,}"); setv(f"MIA_{tag}_nnon", f"{n_non:,}")
    def boot(key):
        P = [sc[s][key + "|mem"] for s in seeds if key + "|mem" in sc[s]]
        Q = [sc[s][key + "|non"] for s in seeds if key + "|non" in sc[s]]
        if not P:
            return None
        return seed_mean_auc_bootstrap(P, Q, CONFIG["mia"]["n_boot"])
    rows = []
    # loss attack
    g = pr[(pr.method == "diffusion") & (pr.attack == "loss")]
    b = boot("diffusion|loss|generator|0")
    setv(f"MIA_{tag}_loss", pm(g.auc)); setv(f"MIA_{tag}_lossm", f3(g.auc.mean()))
    setv(f"MIA_{tag}_lossci", f"{b[1]:.3f}–{b[2]:.3f}" if b else "n/a")
    setv(f"MIA_{tag}_lossmin", f3(g.auc.min())); setv(f"MIA_{tag}_lossmax", f3(g.auc.max()))
    rows.append({"Method": LABEL["diffusion"], "Released artefact": "Generator (white-box)", "Attack": "Loss",
                 "AUC mean ± SD": pm(g.auc), "95% CI": f"{b[1]:.3f}–{b[2]:.3f}" if b else "n/a",
                 "Exact copies of members": "n/a", "Seeds": g.seed.nunique()})
    if "diff_stop_epoch" in g:
        setv(f"{tag}_diff_stop", pm(g.diff_stop_epoch, 0))
    # nearest-neighbour attacks on the released synthetic set
    for m in AUG_METHODS:
        for release, k, suffix in [("released", 1, "nn"), ("released", 5, "nn5"), ("fixed", 1, "nnfix")]:
            g = pr[(pr.method == m) & (pr.attack == "nn") & (pr.release == release) & (pr.k == k)]
            if len(g) == 0:
                continue
            b = boot(f"{m}|nn|{release}|{k}")
            setv(f"MIA_{tag}_{m}_{suffix}", pm(g.auc)); setv(f"MIA_{tag}_{m}_{suffix}m", f3(g.auc.mean()))
            setv(f"MIA_{tag}_{m}_{suffix}ci", f"{b[1]:.3f}–{b[2]:.3f}" if b else "n/a")
            if suffix == "nn":
                setv(f"MIA_{tag}_{m}_copy", f"{100 * g.copy_share.mean():.1f}%")
                setv(f"MIA_{tag}_{m}_nsyn", f"{int(g.n_synth.iloc[0]):,}")
                rows.append({"Method": LABEL[m], "Released artefact": f"Synthetic set ({int(g.n_synth.iloc[0]):,} records)",
                             "Attack": "Nearest neighbour (k=1)", "AUC mean ± SD": pm(g.auc),
                             "95% CI": f"{b[1]:.3f}–{b[2]:.3f}" if b else "n/a",
                             "Exact copies of members": f"{100 * g.copy_share.mean():.1f}%", "Seeds": g.seed.nunique()})
    # classifier-level attack
    if cm is not None:
        for m in METHODS:
            g = cm[cm.method == m]
            if len(g) == 0:
                continue
            b = boot(f"{m}|classifier|released|0")
            setv(f"MIA_{tag}_{m}_clf", pm(g.auc)); setv(f"MIA_{tag}_{m}_clfm", f3(g.auc.mean()))
            setv(f"MIA_{tag}_{m}_clfci", f"{b[1]:.3f}–{b[2]:.3f}" if b else "n/a")
            rows.append({"Method": LABEL[m], "Released artefact": f"Classifier ({g.classifier.iloc[0]})",
                         "Attack": "Confidence (classifier-level)", "AUC mean ± SD": pm(g.auc),
                         "95% CI": f"{b[1]:.3f}–{b[2]:.3f}" if b else "n/a", "Exact copies of members": "n/a",
                         "Seeds": g.seed.nunique()})
    # paired comparison of the NN attack: diffusion against each other method across generator seeds
    nn = pr[(pr.attack == "nn") & (pr.release == "released") & (pr.k == 1)].pivot_table(index="seed", columns="method", values="auc")
    if "diffusion" in nn:
        others = [m for m in AUG_METHODS if m != "diffusion" and m in nn]
        res = {m: paired_test(nn["diffusion"].values, nn[m].values) for m in others}
        adj = holm([res[m]["p"] for m in others]) if others else []
        for m, pa in zip(others, adj):
            setv(f"MIAT_{tag}_{m}_d", fd(res[m])); setv(f"MIAT_{tag}_{m}_ph", fp(pa))
    tab = pd.DataFrame(rows); tab.to_csv(os.path.join(TABLES_DIR, f"{prefix}_privacy.csv"), index=False)
    return pr

if PARTS["G_assemble"]:
    P_ULB = privacy_tables("ulb", "ULB")
    P_IEEE = privacy_tables("ieee", "IEEE")
    P_TMP = privacy_tables("temporal", "TMP")

In [19]:
# ============================================================
# 13. DP, scale and hyperparameter tables
# ============================================================
SIGKEY = {"none": "none", "0.5": "0p5", "1.0": "1", "2.0": "2", "4.0": "4"}
if PARTS["G_assemble"]:
    dp = load_csv("ulb_dp.csv")
    if dp is not None:
        dp["nm"] = dp.noise_multiplier.astype(str)
        rows = []
        setv("DP_NSEEDS", str(dp.seed.nunique())); setv("DP_EPOCHS", str(int(dp.epochs.iloc[0])))
        setv("DP_CLF", str(dp.classifier.iloc[0])); setv("DP_RATIO", fratio(float(dp.ratio.iloc[0])))
        for arch in ["matched", "light"]:
            for nm in ["none", "0.5", "1.0", "2.0", "4.0"]:
                g = dp[(dp.arch == arch) & (dp.nm == nm)]
                if len(g) == 0:
                    continue
                k = f"DP_{arch}_{SIGKEY[nm]}"
                eps = g.epsilon.replace([np.inf], np.nan).mean()
                setv(k + "_eps", "∞ (no DP)" if nm == "none" else f"{eps:.2f}")
                setv(k + "_f1", pm(g.f1)); setv(k + "_f1m", f3(g.f1.mean()))
                setv(k + "_auprc", pm(g.auprc)); setv(k + "_loss", pm(g.auc_loss)); setv(k + "_lossm", f3(g.auc_loss.mean()))
                setv(k + "_nn", pm(g.auc_nn))
                rows.append({"Denoiser": "Primary (LayerNorm, 4 blocks)" if arch == "matched" else "Lightweight (no norm, 2 blocks)",
                             "Noise multiplier σ": "no DP" if nm == "none" else nm,
                             "ε (δ=1e-5)": "∞" if nm == "none" else f"{eps:.2f}", "F1": pm(g.f1), "AUPRC": pm(g.auprc),
                             "Loss-attack AUC": pm(g.auc_loss), "NN-attack AUC": pm(g.auc_nn), "Seeds": g.seed.nunique()})
        pd.DataFrame(rows).to_csv(os.path.join(TABLES_DIR, "ulb_dp.csv"), index=False)

    scl = load_csv("scale_results.csv")
    if scl is not None:
        rows = []
        cap = CONFIG["scale"]["max_epochs"]
        for (ds, n), g in scl.groupby(["dataset", "n"]):
            tag = "ULB" if ds == "ULB" else f"IEEE_{n}"
            rec = {"Dataset": ds, "Training frauds n": n, "Non-members": int(g.n_nonmembers.iloc[0])}
            for rule, key, lab in [("train_loss_rule", "trainrule", "Training-loss rule"), ("heldout", "heldout", "Held-out-loss early stopping")]:
                h = g[g.rule == rule]
                setv(f"SC_{tag}_{key}", pm(h.auc)); setv(f"SC_{tag}_{key}m", f3(h.auc.mean())); setv(f"SC_{tag}_{key}_ep", pm(h.epoch, 0))
                rec[f"{lab}: AUC"] = pm(h.auc); rec[f"{lab}: epoch"] = pm(h.epoch, 0)
            for e in CONFIG["scale"]["checkpoints"]:
                h = g[(g.rule == "fixed") & (g.epoch == e)]
                setv(f"SC_{tag}_e{e}", pm(h.auc)); setv(f"SC_{tag}_e{e}m", f3(h.auc.mean()))
            h = g[(g.rule == "fixed") & (g.epoch == cap)]
            rec[f"Fixed {cap} epochs: AUC"] = pm(h.auc)
            rec["Seeds"] = g.seed.nunique()
            if ds == "ULB":
                setv("SC_ULB_n", str(n))
            rows.append(rec)
        pd.DataFrame(rows).to_csv(os.path.join(TABLES_DIR, "scale_results.csv"), index=False)
        # rank correlation between n and AUC at a fixed epoch budget (IEEE-CIS only)
        ie = scl[(scl.dataset == "IEEE-CIS") & (scl.rule == "fixed") & (scl.epoch == cap)]
        if len(ie) > 3:
            rho, p = stats.spearmanr(ie.n, ie.auc); setv("SC_IEEE_rho", f"{rho:.2f}"); setv("SC_IEEE_rho_p", fp(p))

    # hyperparameter table for the manuscript
    c, d_, cl, dpc_ = CONFIG["ctgan"], CONFIG["diffusion"], CONFIG["classifiers"], CONFIG["dp"]
    hp = [("Diffusion", "Noise schedule", f"linear, T = {d_['timesteps']}, β from {d_['beta_start']} to {d_['beta_end']}"),
          ("Diffusion", "Denoiser", f"residual MLP, width {d_['hidden_dim']}, {d_['n_blocks']} blocks, time embedding {d_['t_dim']}, LayerNorm, SiLU"),
          ("Diffusion", "Optimiser", f"Adam, learning rate {d_['lr']}, batch size {d_['batch_size']}, EMA decay {d_['ema_decay']}"),
          ("Diffusion", "Stopping rule (Eq. 7)", f"window w = max(20, cap/20), tolerance {d_['rel_tol']}, floor {d_['min_epochs']} (ULB) / {CONFIG['ieee']['min_epochs']} (IEEE-CIS), cap {d_['max_epochs']} / {CONFIG['ieee']['max_epochs']}"),
          ("Diffusion", "Loss attack", f"{d_['loss_repeats']} (t, ε) draws per record"),
          ("CTGAN", "Architecture", f"embedding {c['embedding_dim']}, generator {c['generator_dim']}, discriminator {c['discriminator_dim']}, pac {c['pac']}"),
          ("CTGAN", "Training", f"{c['epochs']} epochs, batch {c['batch_size']}, Adam lr {c['generator_lr']} (both), weight decay {c['generator_decay']}"),
          ("SMOTE / ADASYN", "Neighbours", f"k = {CONFIG['resampling']['k_neighbors']}, real majority reference of {CONFIG['resampling']['maj_ref_size']:,} records"),
          ("Logistic regression", "Settings", f"L2, C = {cl['lr_C']}, lbfgs, max_iter {cl['lr_max_iter']}"),
          ("Random forest", "Settings", f"{cl['rf_n_estimators']} trees, max depth {cl['rf_max_depth']}, min samples per leaf {cl['rf_min_samples_leaf']}"),
          ("XGBoost", "Settings", f"{cl['xgb_n_estimators']} trees, depth {cl['xgb_max_depth']}, learning rate {cl['xgb_learning_rate']}, subsample {cl['xgb_subsample']}, column subsample {cl['xgb_colsample_bytree']}"),
          ("MLP", "Settings", f"hidden {cl['mlp_hidden']}, ReLU, Adam lr {cl['mlp_lr']}, {cl['mlp_epochs']} epochs, batch {cl['mlp_batch_size']}, pos_weight = n_neg/n_pos"),
          ("All classifiers", "Decision threshold", f"{cl['decision_threshold']} (fixed) and the validation-tuned threshold"),
          ("DP-SGD", "Settings", f"clip C = {dpc_['max_grad_norm']}, δ = {dpc_['delta']}, σ ∈ {dpc_['noise_multipliers']}, {dpc_['epochs']} epochs, Poisson sampling, RDP accountant"),
          ("Protocol", "Seeds", f"ULB: {len(CONFIG['screen_seeds'])} selection + {len(CONFIG['confirm_seeds'])} confirmation; IEEE-CIS: {len(CONFIG['ieee']['screen_seeds'])} + {len(CONFIG['ieee']['confirm_seeds'])}; DP: {len(dpc_['seeds'])}; scale study: {len(CONFIG['scale']['seeds'])}")]
    pd.DataFrame(hp, columns=["Component", "Item", "Setting"]).to_csv(os.path.join(TABLES_DIR, "hyperparameters.csv"), index=False)

    rt = find_result("runtime_log.json")
    if rt:
        st = json.load(open(rt))["stages"]
        setv("RT_total_h", f"{sum(s['seconds'] for s in st) / 3600:.1f}")

In [20]:
# ============================================================
# 14. Figures (2x2 / 2x3 grids, regenerated from the saved results)
# ============================================================
def sweep_figure(prefix, name, sel):
    s1 = load_csv(f"{prefix}_stage1_validation.csv")
    if s1 is None:
        return
    agg = s1.groupby(["classifier", "method", "ratio"]).f1.agg(["mean", "std"]).reset_index().fillna(0)
    clfs = [c for c in CLASSIFIER_NAMES if c in agg.classifier.unique()]
    ncol = 2 if len(clfs) > 1 else 1; nrow = int(math.ceil(len(clfs) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(6.2 * ncol, 4.6 * nrow), squeeze=False)
    ratios = sorted(agg[agg.method != "none"].ratio.unique())
    for ax, c in zip(axes.ravel(), clfs):
        for m in AUG_METHODS:
            g = agg[(agg.classifier == c) & (agg.method == m)].sort_values("ratio")
            if len(g):
                ax.errorbar(g.ratio, g["mean"], yerr=g["std"], marker="o", capsize=3, color=PALETTE[m], label=LABEL[m])
        b = agg[(agg.classifier == c) & (agg.method == "none")]
        if len(b):
            ax.axhline(b["mean"].iloc[0], color=PALETTE["none"], ls=":", lw=2, label=LABEL["none"])
        ax.set_xscale("log"); ax.set_xticks(ratios); ax.set_xticklabels([f"{r:g}" for r in ratios])
        ax.minorticks_off()
        ax.set_title(c + (" (primary)" if sel and c == sel["primary_classifier"] else ""))
        ax.set_xlabel("Augmentation ratio r (× training frauds)"); ax.set_ylabel("Validation F1 (fraud class)")
    for ax in axes.ravel()[len(clfs):]:
        ax.axis("off")
    h, l = axes.ravel()[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.06))
    fig.tight_layout(rect=(0, 0.04, 1, 1)); save_fig(fig, name)
    FIG_DATA[name] = [("sweep", agg.rename(columns={"mean": "val_f1_mean", "std": "val_f1_sd"}), "sweep")]

def boxplot_figure(prefix, name, sel):
    s2 = load_csv(f"{prefix}_stage2_test.csv")
    if s2 is None or sel is None:
        return
    prim = sel["primary_classifier"]; d = s2[s2.classifier == prim]
    ms = [m for m in METHODS if m in d.method.unique()]
    fig, ax = plt.subplots(figsize=(11, 5.2))
    data = [d[d.method == m].f1.values for m in ms]
    bp = ax.boxplot(data, patch_artist=True, showmeans=True, medianprops=dict(color="black"),
                    meanprops=dict(marker="D", markerfacecolor="white", markeredgecolor="black"))
    for patch, m in zip(bp["boxes"], ms):
        patch.set_facecolor(PALETTE[m]); patch.set_alpha(0.55)
    rng = np.random.RandomState(SEED)
    for i, v in enumerate(data):
        ax.scatter(i + 1 + rng.uniform(-0.08, 0.08, len(v)), v, s=22, color="black", alpha=0.6, zorder=3)
    ax.set_xticks(range(1, len(ms) + 1))
    ax.set_xticklabels([f"{LABEL[m]}\n(r* = {fratio(sel['ratio'].get(m + '|' + prim, 0.0))})" if m != "none" else LABEL[m] for m in ms])
    ax.set_ylabel(f"Test F1 (fraud class), {prim}")
    save_fig(fig, name)
    FIG_DATA[name] = [("per_seed_f1", d[["seed", "method", "ratio", "f1", "f1_tuned", "auprc", "mcc"]], "strip")]

def pca_figure(prefix, name):
    p = find_result(f"{prefix}_pca_fidelity.csv")
    if not p:
        return
    df = pd.read_csv(p)
    ms = [m for m in AUG_METHODS if m in df.method.unique()]
    fig, axes = plt.subplots(2, 3, figsize=(15, 9.5), sharex=True, sharey=True)
    leg = df[df.group == "real legitimate"]; fr = df[df.group == "real fraud"]
    for ax, m in zip(axes.ravel(), ms):
        s = df[(df.group == "synthetic") & (df.method == m)]
        ax.scatter(leg.pc1, leg.pc2, s=6, alpha=0.15, color="#999999", label="Real legitimate")
        ax.scatter(fr.pc1, fr.pc2, s=16, alpha=0.8, color="#0B3C8C", label="Real fraud (training)")
        ax.scatter(s.pc1, s.pc2, s=14, alpha=0.6, marker="x", color=PALETTE[m], label="Synthetic")
        ax.set_title(LABEL[m]); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    last = axes.ravel()[-1]; last.axis("off")
    from matplotlib.lines import Line2D
    handles = [Line2D([], [], ls="", marker="o", color="#999999", label="Real legitimate"),
               Line2D([], [], ls="", marker="o", color="#0B3C8C", label="Real fraud (training)")] + \
              [Line2D([], [], ls="", marker="x", color=PALETTE[m], label=f"Synthetic: {LABEL[m]}") for m in ms]
    last.legend(handles=handles, loc="center", fontsize=12)
    save_fig(fig, name)
    FIG_DATA[name] = [("pca_points", df, "pca")]

def mia_figure(prefix, tag, name):
    pr = load_csv(f"{prefix}_privacy.csv"); cm = load_csv(f"{prefix}_classifier_mia.csv"); sc = load_npz_scores(prefix)
    if pr is None or not sc:
        return
    s0 = sorted(sc)[0]
    lm, ln = -sc[s0]["diffusion|loss|generator|0|mem"], -sc[s0]["diffusion|loss|generator|0|non"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))
    bins = np.histogram_bin_edges(np.r_[lm, ln], bins=30)
    axes[0].hist(lm, bins=bins, alpha=0.65, density=True, color=PALETTE["member"], label=f"Members (training frauds, n={len(lm)})")
    axes[0].hist(ln, bins=bins, alpha=0.65, density=True, color=PALETTE["nonmember"], label=f"Non-members (test frauds, n={len(ln)})")
    axes[0].set_xlabel("Per-record diffusion loss"); axes[0].set_ylabel("Density")
    axes[0].set_title(f"(a) Loss attack, generator seed {s0}"); axes[0].legend(fontsize=10)
    rows = []
    g = pr[(pr.method == "diffusion") & (pr.attack == "loss")]
    rows.append({"label": "Diffusion\nloss (white-box)", "method": "diffusion", "attack": "loss", "mean": g.auc.mean(), "sd": g.auc.std()})
    for m in AUG_METHODS:
        g = pr[(pr.method == m) & (pr.attack == "nn") & (pr.release == "released") & (pr.k == 1)]
        if len(g):
            rows.append({"label": f"{LABEL[m].replace(' (DDPM)', '')}\nNN (synthetic set)", "method": m, "attack": "nn",
                         "mean": g.auc.mean(), "sd": g.auc.std()})
    if cm is not None:
        for m in METHODS:
            g = cm[cm.method == m]
            if len(g):
                rows.append({"label": f"{LABEL[m].replace(' (DDPM)', '').replace('No augmentation', 'No aug.')}\nclassifier", "method": m,
                             "attack": "classifier", "mean": g.auc.mean(), "sd": g.auc.std()})
    R = pd.DataFrame(rows)
    x = np.arange(len(R))
    hatch = {"loss": "", "nn": "", "classifier": "//"}
    for i, r in R.iterrows():
        axes[1].bar(i, r["mean"], yerr=r["sd"], capsize=3, color=PALETTE[r["method"]], alpha=0.85 if r["attack"] != "classifier" else 0.45,
                    hatch=hatch[r["attack"]], edgecolor="black", linewidth=0.5)
    first = sc[s0]; lo_c, hi_c = chance_band(len(first["diffusion|loss|generator|0|mem"]), len(first["diffusion|loss|generator|0|non"]), CONFIG["mia"]["n_perm"])
    axes[1].axhspan(lo_c, hi_c, color="grey", alpha=0.2, label="95% chance band")
    axes[1].axhline(0.5, color="black", ls=":", lw=1)
    axes[1].set_xticks(x); axes[1].set_xticklabels(R.label, rotation=90, fontsize=9)
    axes[1].set_ylim(0.3, 1.02); axes[1].set_ylabel("Membership-inference AUC")
    axes[1].set_title("(b) All attacks and released artefacts"); axes[1].legend(loc="upper right", fontsize=10)
    save_fig(fig, name)
    hist = pd.DataFrame({"bin_left": bins[:-1], "bin_right": bins[1:],
                         "members_density": np.histogram(lm, bins=bins, density=True)[0],
                         "nonmembers_density": np.histogram(ln, bins=bins, density=True)[0]})
    FIG_DATA[name] = [("a_loss_hist", hist, "hist"), ("b_auc_bars", R.assign(label=R.label.str.replace("\n", " ")), "bars")]

def dp_figure(name):
    dp = load_csv("ulb_dp.csv")
    if dp is None:
        return
    dp["nm"] = dp.noise_multiplier.astype(str)
    agg = dp.groupby(["arch", "nm"]).agg(eps=("epsilon", "mean"), f1=("f1", "mean"), f1_sd=("f1", "std"),
                                         loss=("auc_loss", "mean"), loss_sd=("auc_loss", "std"),
                                         nn=("auc_nn", "mean"), nn_sd=("auc_nn", "std")).reset_index()
    finite = agg[np.isfinite(agg.eps)]
    xmax = finite.eps.max() * 4 if len(finite) else 1000
    agg["x"] = np.where(np.isfinite(agg.eps), agg.eps, xmax)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
    style = {"matched": ("o", "-", "Primary denoiser (DP-SGD)"), "light": ("s", "--", "Lightweight denoiser (DP-SGD)")}
    s2 = load_csv("ulb_stage2_test.csv"); sel = load_sel("ulb")
    for arch, (mk, ls, lab) in style.items():
        g = agg[agg.arch == arch].sort_values("x")
        if not len(g):
            continue
        col = PALETTE["diffusion"] if arch == "matched" else PALETTE["adasyn"]
        axes[0].errorbar(g.x, g.f1, yerr=g.f1_sd.fillna(0), marker=mk, ls=ls, capsize=3, color=col, label=lab)
        axes[1].errorbar(g.x, g.loss, yerr=g.loss_sd.fillna(0), marker=mk, ls=ls, capsize=3, color=col, label=lab + ": loss attack")
        axes[1].errorbar(g.x, g.nn, yerr=g.nn_sd.fillna(0), marker=mk, ls=":", capsize=3, color=col, alpha=0.6, label=lab + ": NN attack")
    if s2 is not None and sel is not None:
        prim = sel["primary_classifier"]; d = s2[s2.classifier == prim]
        axes[0].axhline(d[d.method == "diffusion"].f1.mean(), color=PALETTE["diffusion"], ls="-.", lw=1.5, label="Undefended, early-stopped (Part A)")
        axes[0].axhline(d[d.method == "none"].f1.mean(), color=PALETTE["none"], ls=":", lw=2, label="No augmentation")
    pr = load_csv("ulb_privacy.csv")
    if pr is not None:
        axes[1].axhline(pr[(pr.method == "diffusion") & (pr.attack == "loss")].auc.mean(), color=PALETTE["diffusion"], ls="-.", lw=1.5,
                        label="Undefended, early-stopped: loss attack")
    axes[1].axhline(0.5, color="black", ls=":", lw=1)
    for ax in axes:
        ax.set_xscale("log"); ax.set_xlabel("Privacy budget ε (log scale)")
        ticks = sorted(set(np.round(finite.eps, 2))) + [xmax]
        ax.set_xticks(ticks); ax.set_xticklabels([f"{t:g}" for t in ticks[:-1]] + ["no DP"]); ax.minorticks_off()
        ax.axvline(np.sqrt(finite.eps.max() * xmax) if len(finite) else xmax / 2, color="grey", lw=0.8, ls="--")
    axes[0].set_ylabel("Test F1 (fraud class)"); axes[1].set_ylabel("Membership-inference AUC")
    axes[0].set_title("(a) Utility"); axes[1].set_title("(b) Empirical leakage")
    axes[0].legend(fontsize=9, loc="best", frameon=True, framealpha=0.9); axes[1].legend(fontsize=8.5, loc="best", frameon=True, framealpha=0.9)
    save_fig(fig, name)
    FIG_DATA[name] = [("dp_summary", agg, "dp")]

def scale_figure(name):
    scl = load_csv("scale_results.csv")
    if scl is None:
        return
    fx = scl[scl.rule == "fixed"].groupby(["dataset", "n", "epoch"]).auc.agg(["mean", "std"]).reset_index()
    st = scl[scl.rule != "fixed"].groupby(["dataset", "n", "rule"]).agg(auc=("auc", "mean"), auc_sd=("auc", "std"),
                                                                        epoch=("epoch", "mean")).reset_index()
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
    cmap = plt.get_cmap("viridis")
    ieee_n = sorted(fx[fx.dataset == "IEEE-CIS"].n.unique())
    for i, n in enumerate(ieee_n):
        g = fx[(fx.dataset == "IEEE-CIS") & (fx.n == n)]
        axes[0].errorbar(g.epoch, g["mean"], yerr=g["std"].fillna(0), marker="o", capsize=3,
                         color=cmap(i / max(1, len(ieee_n) - 1) * 0.85), label=f"IEEE-CIS, n = {n:,}")
    g = fx[fx.dataset == "ULB"]
    if len(g):
        axes[0].errorbar(g.epoch, g["mean"], yerr=g["std"].fillna(0), marker="s", ls="--", capsize=3, color="black",
                         label=f"ULB, n = {int(g.n.iloc[0])}")
    axes[0].axhline(0.5, color="grey", ls=":"); axes[0].set_xscale("log")
    axes[0].set_xticks(CONFIG["scale"]["checkpoints"]); axes[0].set_xticklabels([str(c) for c in CONFIG["scale"]["checkpoints"]])
    axes[0].minorticks_off()
    axes[0].set_xlabel("Training epochs (log scale)"); axes[0].set_ylabel("Loss-attack AUC")
    axes[0].set_title("(a) Leakage against training length"); axes[0].legend(fontsize=10)
    cap = CONFIG["scale"]["max_epochs"]
    rule_lab = {"train_loss_rule": ("Training-loss rule (Eq. 7)", "o", "-"), "heldout": ("Held-out-loss early stopping", "s", "--")}
    for rule, (lab, mk, ls) in rule_lab.items():
        g = st[(st.dataset == "IEEE-CIS") & (st.rule == rule)].sort_values("n")
        axes[1].errorbar(g.n, g.auc, yerr=g.auc_sd.fillna(0), marker=mk, ls=ls, capsize=3, label=f"IEEE-CIS: {lab}")
    g = fx[(fx.dataset == "IEEE-CIS") & (fx.epoch == cap)].sort_values("n")
    axes[1].errorbar(g.n, g["mean"], yerr=g["std"].fillna(0), marker="^", ls=":", capsize=3, label=f"IEEE-CIS: fixed {cap} epochs")
    for rule, (lab, mk, ls) in rule_lab.items():
        g = st[(st.dataset == "ULB") & (st.rule == rule)]
        if len(g):
            axes[1].scatter(g.n, g.auc, marker=mk, s=90, facecolors="none", edgecolors="black", label=f"ULB: {lab}")
    axes[1].axhline(0.5, color="grey", ls=":"); axes[1].set_xscale("log")
    nt = sorted(set(int(v) for v in st.n.unique()))
    axes[1].set_xticks(nt); axes[1].set_xticklabels([f"{int(v):,}" for v in nt]); axes[1].minorticks_off()
    axes[1].set_xlabel("Training frauds n (log scale)"); axes[1].set_ylabel("Loss-attack AUC")
    axes[1].set_title("(b) Leakage against data scale"); axes[1].legend(fontsize=9)
    save_fig(fig, name)
    FIG_DATA[name] = [("a_auc_vs_epoch", fx, "scale_epoch"), ("b_auc_at_stop", st, "scale_n")]

def cross_dataset_figure(name):
    rows = []
    for prefix, ds in [("ulb", "ULB"), ("ieee", "IEEE-CIS")]:
        pr = load_csv(f"{prefix}_privacy.csv")
        if pr is None:
            continue
        n_mem = DATASET_STATS.get("ULB_train_fraud" if ds == "ULB" else "IEEE_train_fraud", "")
        g = pr[(pr.method == "diffusion") & (pr.attack == "loss")]
        rows.append({"dataset": f"{ds} (n = {n_mem})", "attack": "Diffusion: loss", "mean": g.auc.mean(), "sd": g.auc.std()})
        for m in AUG_METHODS:
            g = pr[(pr.method == m) & (pr.attack == "nn") & (pr.release == "released") & (pr.k == 1)]
            if len(g):
                rows.append({"dataset": f"{ds} (n = {n_mem})", "attack": f"{LABEL[m].replace(' (DDPM)', '')}: NN",
                             "mean": g.auc.mean(), "sd": g.auc.std()})
    if not rows:
        return
    R = pd.DataFrame(rows); attacks = list(dict.fromkeys(R.attack)); dsets = list(dict.fromkeys(R.dataset))
    fig, ax = plt.subplots(figsize=(12, 5.2)); w = 0.8 / len(dsets); x = np.arange(len(attacks))
    for i, ds in enumerate(dsets):
        g = R[R.dataset == ds].set_index("attack").reindex(attacks)
        col = PALETTE["ULB"] if ds.startswith("ULB") else PALETTE["IEEE-CIS"]
        bars = ax.bar(x + (i - (len(dsets) - 1) / 2) * w, g["mean"], w, yerr=g["sd"].fillna(0), capsize=3, color=col, label=ds)
        for b_, v, sd in zip(bars, g["mean"], g["sd"].fillna(0)):
            if np.isfinite(v):
                ax.annotate(f"{v:.3f}", (b_.get_x() + b_.get_width() / 2, v + sd), ha="center", va="bottom", fontsize=9,
                            xytext=(0, 3), textcoords="offset points")
    ax.axhline(0.5, color="black", ls=":", lw=1, label="Random guessing")
    ax.set_xticks(x); ax.set_xticklabels(attacks, rotation=20, ha="right"); ax.set_ylim(0.3, 1.05)
    ax.set_ylabel("Membership-inference AUC"); ax.legend(fontsize=10)
    save_fig(fig, name)
    FIG_DATA[name] = [("cross_dataset", R, "cross")]

def shap_figure(name, top=15):
    sh = load_csv("ulb_shap.csv")
    if sh is None:
        return
    base = sh[sh.method == "none"].set_index("feature").mean_abs_shap
    aug = sh[sh.method == "diffusion"].set_index("feature").mean_abs_shap
    order = base.sort_values(ascending=False).index[:top][::-1]
    fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True)
    axes[0].barh(order, base.reindex(order), color=PALETTE["none"]); axes[0].set_title("(a) No augmentation")
    axes[1].barh(order, aug.reindex(order), color=PALETTE["diffusion"]); axes[1].set_title("(b) Diffusion-augmented")
    for ax in axes:
        ax.set_xlabel("Mean |SHAP value|")
    rho = stats.spearmanr(base.reindex(base.index), aug.reindex(base.index)).correlation
    setv("SHAP_rho", f"{rho:.2f}")
    top_b, top_a = list(base.sort_values(ascending=False).index[:5]), list(aug.sort_values(ascending=False).index[:5])
    setv("SHAP_top5_overlap", str(len(set(top_b) & set(top_a))))
    save_fig(fig, name)
    FIG_DATA[name] = [("shap", pd.DataFrame({"feature": base.index, "none": base.values, "diffusion": aug.reindex(base.index).values}), "shap")]

def loss_curve_figure(name):
    frames = []
    for prefix, ds in [("ulb", "ULB"), ("ieee", "IEEE-CIS")]:
        d = load_csv(f"{prefix}_diffusion_loss_curves.csv")
        if d is not None:
            frames.append(d.assign(dataset=ds))
    if not frames:
        return
    D = pd.concat(frames)
    fig, axes = plt.subplots(1, len(frames), figsize=(7 * len(frames), 4.6), squeeze=False)
    for ax, (ds, g) in zip(axes.ravel(), D.groupby("dataset")):
        for s, h in g.groupby("seed"):
            ax.plot(h.epoch, h.loss, lw=1, alpha=0.6)
        ax.set_title(ds); ax.set_xlabel("Epoch"); ax.set_ylabel("Training loss (noise-prediction MSE)")
    save_fig(fig, name)

if PARTS["G_assemble"]:
    sweep_figure("ulb", "fig03_utility_sweep_ulb", SEL["ulb"])
    boxplot_figure("ulb", "fig04_seed_f1_ulb", SEL["ulb"])
    pca_figure("ulb", "fig05_pca_fidelity_ulb")
    mia_figure("ulb", "ULB", "fig06_mia_audit_ulb")
    dp_figure("fig07_dp_sweep_ulb")
    scale_figure("fig08_scale_and_epochs")
    cross_dataset_figure("fig09_cross_dataset_mia")
    sweep_figure("ieee", "fig10_utility_sweep_ieee", SEL["ieee"])
    shap_figure("fig11_shap_ulb")
    loss_curve_figure("figS1_diffusion_training_loss")

saved fig03_utility_sweep_ulb
saved fig04_seed_f1_ulb
saved fig05_pca_fidelity_ulb
saved fig06_mia_audit_ulb
saved fig07_dp_sweep_ulb
saved fig08_scale_and_epochs
saved fig09_cross_dataset_mia
saved fig10_utility_sweep_ieee
saved fig11_shap_ulb
saved figS1_diffusion_training_loss


In [21]:
# ============================================================
# 15. Editable figure source: every plotted table + native Excel charts
# ============================================================
def write_chart_workbook(path):
    import xlsxwriter
    wb = xlsxwriter.Workbook(path)
    bold = wb.add_format({"bold": True})
    wb.add_worksheet("README").write_column(0, 0, [
        "Editable source data for every chart in the revised manuscript.",
        "Each sheet holds the numbers plotted in one figure panel, with a native Excel chart next to the data.",
        "Charts can be restyled in Excel and pasted into Word as editable charts.",
        "Box plots are shown as per-seed strip charts (Excel's box-and-whisker chart is not scriptable).",
        f"Generated {time.strftime('%Y-%m-%d %H:%M')} from {RESULTS_DIR}"])
    def put(ws, df, r0=0, c0=0):
        for j, col in enumerate(df.columns):
            ws.write(r0, c0 + j, str(col), bold)
        for i, row in enumerate(df.itertuples(index=False)):
            for j, v in enumerate(row):
                if isinstance(v, (float, np.floating)) and not np.isfinite(v):
                    ws.write_blank(r0 + 1 + i, c0 + j, None)
                elif isinstance(v, (int, float, np.integer, np.floating)):
                    ws.write_number(r0 + 1 + i, c0 + j, float(v))
                else:
                    ws.write(r0 + 1 + i, c0 + j, str(v))
        return r0 + 1, r0 + len(df)
    for fig_name, items in FIG_DATA.items():
        for sub, df, kind in items:
            title = (fig_name.split("_")[0] + "_" + sub)[:31]
            ws = wb.add_worksheet(title)
            if kind == "sweep":
                col0 = 0; charts = 0
                for c, g in df.groupby("classifier"):
                    piv = g.pivot_table(index="ratio", columns="method", values="val_f1_mean").reset_index()
                    sd = g.pivot_table(index="ratio", columns="method", values="val_f1_sd").reset_index()
                    ws.write(0, col0, c, bold)
                    r1, r2 = put(ws, piv, 1, col0); put(ws, sd.add_suffix("_sd"), 1, col0 + len(piv.columns) + 1)
                    ch = wb.add_chart({"type": "scatter", "subtype": "straight_with_markers"})
                    for j, m in enumerate(piv.columns[1:]):
                        sdcol = col0 + len(piv.columns) + 1 + 1 + j
                        ch.add_series({"name": LABEL.get(m, m), "categories": [title, r1 + 1, col0, r2 + 1, col0],
                                       "values": [title, r1 + 1, col0 + 1 + j, r2 + 1, col0 + 1 + j],
                                       "y_error_bars": {"type": "custom", "plus_values": [title, r1 + 1, sdcol, r2 + 1, sdcol],
                                                        "minus_values": [title, r1 + 1, sdcol, r2 + 1, sdcol]},
                                       "line": {"color": PALETTE.get(m, "#000000")},
                                       "marker": {"type": "circle", "fill": {"color": PALETTE.get(m, "#000000")}}})
                    ch.set_title({"name": c}); ch.set_x_axis({"name": "Augmentation ratio", "log_base": 10})
                    ch.set_y_axis({"name": "Validation F1"}); ch.set_size({"width": 560, "height": 340})
                    ws.insert_chart(len(piv) + 4 + charts * 18, 0, ch); charts += 1
                    col0 += 2 * len(piv.columns) + 3
            elif kind in ("strip", "pca", "hist", "bars", "dp", "scale_epoch", "scale_n", "cross", "shap"):
                r1, r2 = put(ws, df.reset_index(drop=True))
                cols = list(df.columns)
                def cidx(name):
                    return cols.index(name)
                if kind == "strip":
                    ch = wb.add_chart({"type": "scatter"})
                    D = df.reset_index(drop=True)
                    ws.write(0, len(cols) + 1, "method_index", bold)
                    for i, m in enumerate(D.method):
                        ws.write_number(1 + i, len(cols) + 1, METHODS.index(m) + 1 if m in METHODS else 0)
                    ch.add_series({"name": "Per-seed F1", "categories": [title, r1, len(cols) + 1, r2, len(cols) + 1],
                                   "values": [title, r1, cidx("f1"), r2, cidx("f1")], "marker": {"type": "circle"}})
                    ch.set_x_axis({"name": "Method (1 = none, 2 = ROS, 3 = SMOTE, 4 = ADASYN, 5 = CTGAN, 6 = diffusion)"})
                    ch.set_y_axis({"name": "Test F1"})
                elif kind == "pca":
                    ch = wb.add_chart({"type": "scatter"})
                    D = df.reset_index(drop=True)
                    for key, g in D.groupby(["group", "method"], dropna=False, sort=False):
                        idx = g.index
                        if len(idx) == 0 or (idx.max() - idx.min() + 1) != len(idx):
                            continue
                        nm = key[0] if not isinstance(key[1], str) or key[1] == "" else f"synthetic {key[1]}"
                        ch.add_series({"name": nm, "categories": [title, r1 + idx.min(), cidx("pc1"), r1 + idx.max(), cidx("pc1")],
                                       "values": [title, r1 + idx.min(), cidx("pc2"), r1 + idx.max(), cidx("pc2")],
                                       "marker": {"type": "circle", "size": 3}})
                    ch.set_x_axis({"name": "PC1"}); ch.set_y_axis({"name": "PC2"})
                elif kind == "hist":
                    ch = wb.add_chart({"type": "column", "subtype": "clustered"})
                    for col in ("members_density", "nonmembers_density"):
                        ch.add_series({"name": col, "categories": [title, r1, 0, r2, 0], "values": [title, r1, cidx(col), r2, cidx(col)]})
                    ch.set_x_axis({"name": "Per-record loss (left bin edge)"}); ch.set_y_axis({"name": "Density"})
                elif kind in ("bars", "cross"):
                    ch = wb.add_chart({"type": "column"})
                    if kind == "bars":
                        ch.add_series({"name": "AUC", "categories": [title, r1, cidx("label"), r2, cidx("label")],
                                       "values": [title, r1, cidx("mean"), r2, cidx("mean")],
                                       "y_error_bars": {"type": "custom", "plus_values": [title, r1, cidx("sd"), r2, cidx("sd")],
                                                        "minus_values": [title, r1, cidx("sd"), r2, cidx("sd")]}})
                    else:
                        D = df.reset_index(drop=True)
                        for ds, g in D.groupby("dataset", sort=False):
                            ch.add_series({"name": ds, "categories": [title, r1 + g.index.min(), cidx("attack"), r1 + g.index.max(), cidx("attack")],
                                           "values": [title, r1 + g.index.min(), cidx("mean"), r1 + g.index.max(), cidx("mean")],
                                           "y_error_bars": {"type": "custom",
                                                            "plus_values": [title, r1 + g.index.min(), cidx("sd"), r1 + g.index.max(), cidx("sd")],
                                                            "minus_values": [title, r1 + g.index.min(), cidx("sd"), r1 + g.index.max(), cidx("sd")]}})
                    ch.set_y_axis({"name": "Membership-inference AUC", "min": 0.3, "max": 1.0})
                elif kind == "dp":
                    ch = wb.add_chart({"type": "scatter", "subtype": "straight_with_markers"})
                    D = df.reset_index(drop=True)
                    for arch, g in D.groupby("arch", sort=False):
                        for ycol in ("f1", "loss"):
                            ch.add_series({"name": f"{arch}: {ycol}", "categories": [title, r1 + g.index.min(), cidx("x"), r1 + g.index.max(), cidx("x")],
                                           "values": [title, r1 + g.index.min(), cidx(ycol), r1 + g.index.max(), cidx(ycol)]})
                    ch.set_x_axis({"name": "ε (log scale; largest x = no DP)", "log_base": 10}); ch.set_y_axis({"name": "F1 / AUC"})
                elif kind == "scale_epoch":
                    ch = wb.add_chart({"type": "scatter", "subtype": "straight_with_markers"})
                    D = df.reset_index(drop=True)
                    for key, g in D.groupby(["dataset", "n"], sort=False):
                        ch.add_series({"name": f"{key[0]} n={key[1]}", "categories": [title, r1 + g.index.min(), cidx("epoch"), r1 + g.index.max(), cidx("epoch")],
                                       "values": [title, r1 + g.index.min(), cidx("mean"), r1 + g.index.max(), cidx("mean")]})
                    ch.set_x_axis({"name": "Epochs", "log_base": 10}); ch.set_y_axis({"name": "Loss-attack AUC"})
                elif kind == "scale_n":
                    ch = wb.add_chart({"type": "scatter", "subtype": "straight_with_markers"})
                    D = df.reset_index(drop=True)
                    for key, g in D.groupby(["dataset", "rule"], sort=False):
                        ch.add_series({"name": f"{key[0]} {key[1]}", "categories": [title, r1 + g.index.min(), cidx("n"), r1 + g.index.max(), cidx("n")],
                                       "values": [title, r1 + g.index.min(), cidx("auc"), r1 + g.index.max(), cidx("auc")]})
                    ch.set_x_axis({"name": "Training frauds n", "log_base": 10}); ch.set_y_axis({"name": "Loss-attack AUC"})
                elif kind == "shap":
                    ch = wb.add_chart({"type": "bar"})
                    for col in ("none", "diffusion"):
                        ch.add_series({"name": col, "categories": [title, r1, 0, r2, 0], "values": [title, r1, cidx(col), r2, cidx(col)]})
                    ch.set_x_axis({"name": "Mean |SHAP|"})
                ch.set_size({"width": 720, "height": 420})
                ws.insert_chart(1, len(cols) + 3, ch)
    wb.close()
    print("Chart workbook ->", path)

if PARTS["G_assemble"]:
    write_chart_workbook(rpath("figure_source_data.xlsx"))

Chart workbook -> /kaggle/working/revision_results/figure_source_data.xlsx


In [22]:
# ============================================================
# 16. Manuscript values (placeholder -> number) and a readable summary
# ============================================================
if PARTS["G_assemble"]:
    json.dump(VALUES, open(rpath("manuscript_values.json"), "w"), indent=1, ensure_ascii=False)
    with open(rpath("RESULTS_SUMMARY.md"), "w") as f:
        f.write(f"# Revision results ({RUN_MODE})\n\n")
        for t in sorted(glob.glob(os.path.join(TABLES_DIR, "*.csv"))):
            f.write(f"## {os.path.basename(t)}\n\n{pd.read_csv(t).to_markdown(index=False)}\n\n")
        f.write("## All manuscript values\n\n")
        for k in sorted(VALUES):
            f.write(f"- `{k}`: {VALUES[k]}\n")
    print(f"{len(VALUES)} manuscript values written to", rpath("manuscript_values.json"))
    for t in sorted(glob.glob(os.path.join(TABLES_DIR, "*.csv"))):
        print("\n==", os.path.basename(t)); print(pd.read_csv(t).to_string(index=False))
    print("\nFigures:", sorted(os.listdir(FIGURES_DIR)))
    # zip everything for a single download
    import zipfile
    with zipfile.ZipFile(os.path.join(WORK_DIR, "revision_outputs.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        for folder in (RESULTS_DIR, FIGURES_DIR):
            for root, _, files in os.walk(folder):
                for fn in files:
                    full = os.path.join(root, fn); z.write(full, os.path.relpath(full, WORK_DIR))
    print("Download:", os.path.join(WORK_DIR, "revision_outputs.zip"))

889 manuscript values written to /kaggle/working/revision_results/manuscript_values.json

== hyperparameters.csv
          Component                  Item                                                                                         Setting
          Diffusion        Noise schedule                                                          linear, T = 200, β from 0.0001 to 0.02
          Diffusion              Denoiser                          residual MLP, width 256, 4 blocks, time embedding 128, LayerNorm, SiLU
          Diffusion             Optimiser                                       Adam, learning rate 0.001, batch size 64, EMA decay 0.995
          Diffusion Stopping rule (Eq. 7)   window w = max(20, cap/20), tolerance 0.02, floor 800 (ULB) / 500 (IEEE-CIS), cap 3000 / 1500
          Diffusion           Loss attack                                                                      20 (t, ε) draws per record
              CTGAN          Architecture                  